# Bước 3 — Kiểm tra chất lượng dữ liệu thô của bảng anime và bảng user

**Mục đích.** Bước 1 so sánh dữ liệu thô với mô tả của tác giả. Bước này nhìn vào bên trong các cột mà câu hỏi nghiên cứu sẽ phụ thuộc vào, và sửa vấn đề username `null` đã phát hiện ở bước 1.

**Cách chạy trên Kaggle.** Thêm *MyAnimeList Dataset* làm input rồi chạy tất cả các cell. Phần 1–3 chỉ đọc hai bảng nhỏ, mất vài giây. Phần 4 đọc hai cột của bảng list 80M dòng, mất vài phút, và phần 7B đọc nó thêm một lần nữa (khoảng 4 phút); đặt `RUN_LISTS = False` trong cell setup để bỏ qua cả hai.

**Cách đọc output.** Mỗi phần kết thúc bằng một cell *Nhận xét*: trước hết nêu output cho thấy gì, rồi mới diễn giải.

| Phần | Câu hỏi |
|---|---|
| 1 | Những giá trị thật nào bị pandas âm thầm biến thành `NaN`? |
| 2 | Bảng anime: placeholder, giá trị không thể có, `score` đáng tin đến đâu, mô hình dự đoán trước khi phát sóng dùng được gì? |
| 3 | Bảng user: placeholder, tuổi và ngày không thể có, các số tổng hợp đáng ngờ |
| 4 | Số đếm theo từng user trong `UserList` có khớp với các dòng trong `UserAnimeList` không? |
| 5 | Những user không có dòng nào trong bảng list: họ có giá trị khác không (nếu có thì bảng list đang thiếu họ)? |
| 6 | Kiểm tra cho rõ: những con số lần chạy đầu chưa in, và các giả thuyết chưa từng được kiểm tra |
| 7 | Những kiểm tra mà phần 1 đến 6 chưa làm: bảng list, các cột số, username, trùng lặp và mâu thuẫn trong cùng một dòng |


**Bước này nằm ở đâu.** Bước 3 trong thứ tự khám phá (xem [docs/exploration_order.md](../docs/exploration_order.md)). Notebook này chỉ **tìm và đếm** vấn đề. Nó không làm sạch gì và không so sánh các cột bằng thống kê. Quy tắc làm sạch được quyết định ở bước 4, dựa trên bằng chứng tìm được ở đây.


In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 4, which reads 80M rows

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 1. Giá trị thật bị pandas biến thành `NaN`

Mặc định pandas đọc các chuỗi như `null`, `NA` hay `None` là giá trị thiếu. Ở bước 1 điều này làm username `null` biến mất. Ở đây mọi ô của hai bảng nhỏ được đọc dưới dạng văn bản thô, và các ô mà pandas sẽ biến thành `NaN` được đếm theo từng cột. Sau đó cả hai bảng được nạp lại, chỉ coi chuỗi rỗng là giá trị thiếu.


### 1a. Nhìn qua các giá trị trước khi đếm những từ thiếu bị giấu (MỚI, thêm sau lần chạy đầu)

**Vì sao làm bước này trước.** Phía dưới, chúng ta đếm những từ mà pandas giấu (`null`, `N/A`, `None`, ...) bằng một danh sách từ do chính mình viết. Ở phần 2 chúng ta cũng đếm placeholder bằng một danh sách tự viết (`Unknown`, `Not available`, `None`, `?`, `0000-00-00`). Cả hai danh sách đều xuất phát từ những gì tình cờ nhìn thấy, nên có thể bỏ sót từ khác, và chúng chỉ tìm được ô **bằng đúng** một trong các từ đó. Một ô như `Fridays at Unknown` sẽ không bị tìm thấy. Vì vậy, trước khi đếm, ta để dữ liệu tự cho thấy có những giá trị giống placeholder nào.

**Các cell làm gì.** Cell đầu nạp hai bảng nhỏ bằng cách nạp an toàn. Với mỗi cột văn bản của bảng anime và bảng user, nó đếm ba thứ và in các giá trị đằng sau mỗi số đếm:
- **Ứng viên chính xác:** ô chỉ gồm một từ giống placeholder, một ký hiệu hoặc số 0 (`?`, `-`, `0`, `none`, `null`, `n/a`, `unknown`, `tba`, `tbd`, `undefined`, `not available`, `0000-00-00`, chỉ có khoảng trắng), không phân biệt hoa thường.
- **Ứng viên một phần:** văn bản dài hơn có chứa `unknown`, `not available`, `n/a`, `tba` hoặc `tbd` (ví dụ `Fridays at Unknown`).
- **Văn bản rất ngắn:** từ hai ký tự trở xuống, thường là lỗi gõ hoặc ký hiệu.

Cell thứ hai in năm giá trị xuất hiện nhiều nhất của mọi cột văn bản để ta tự đọc bằng mắt: placeholder thường là một trong những giá trị phổ biến nhất của cột.

**Giới hạn.** Cách này vẫn chỉ tìm những mẫu mà ta nghĩ ra. Placeholder viết theo cách bất ngờ (`idk`, `nothing`, `Earth` trong `location`) chỉ tìm được bằng cách đọc các giá trị phổ biến nhất. Bảng list (80 triệu dòng) không được quét ở đây; các cột văn bản của nó là `my_tags` (văn bản tự do) và hai cột ngày, mà phần 3 của notebook 02 đã xem.

**Đọc từng kết quả trước khi gọi nó là placeholder.** Việc quét chỉ tìm ứng viên. Ví dụ username `null` là một user thật, nên nó sẽ hiện ra như ứng viên chính xác trong `username` dù không phải placeholder.

**Làm gì với kết quả.** Nếu output cho thấy những từ không có trong danh sách, ta thêm chúng vào danh sách bên dưới và trong phần 2 trước khi đếm.


In [2]:
# NEW: placeholder-like cells in the text columns of the anime and user tables
import re

na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)

CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
PARTIAL = r"unknown|not available|n/a|tba|tbd"

def scan_text(df, label):
    print("=====", label)
    rows, details = [], {}
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        exact = s[s.str.match(CANDIDATE, flags=re.IGNORECASE)]
        partial = s[s.str.contains(PARTIAL, flags=re.IGNORECASE, regex=True)]
        partial = partial[~partial.index.isin(exact.index)]
        short = s[(s.str.len() <= 2) & ~s.index.isin(exact.index)]
        rows.append({"column": col, "non_missing": len(s), "exact": len(exact), "partial": len(partial), "short": len(short)})
        details[col] = (exact, partial, short)
    table = pd.DataFrame(rows).set_index("column")
    hits = table[(table[["exact", "partial", "short"]] > 0).any(axis=1)]
    print(hits.sort_values("exact", ascending=False).to_string())
    for col in hits.index:
        exact, partial, short = details[col]
        print(f"\n{col}")
        if len(exact): print("  exact  :", exact.value_counts().head(6).to_dict())
        if len(partial): print("  partial:", partial.value_counts().head(6).to_dict())
        if len(short): print("  short  :", short.value_counts().head(6).to_dict())

scan_text(anime, "anime table")
print()
scan_text(users, "user table")


===== anime table
                non_missing  exact  partial  short
column                                            
source                14478   4210        0      0
broadcast              4271   2241       78      0
rating                14478    544        0      0
duration              14478    405        0      0
aired_string          14478    223        0      0
type                  14478     30        0   4271
title_synonyms         8937      0       16     11
title_japanese        14443      0        1    165
title                 14478      0        2     12
title_english          5724      0        9      3
background             1057      0        2      0
related               14478      0        1   4515
producer               8288      0        9      6
studio                 8544      0        0      3
opening_theme         14478      0        4   9784
ending_theme          14478      0        5   8807

title
  partial: {'The Epic Of ZektBach': 1, 'GetBackers': 1}
 

In [3]:
# NEW: the five most frequent values of every text column (read them: placeholders are often among them)
def top_values(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        counts = df[col].value_counts().head(n)
        print(f"{col}: " + " | ".join(f"{str(v)[:30]!r} {c:,}" for v, c in counts.items()))

top_values(anime, "anime table")
print()
top_values(users, "user table")


===== anime table
title: 'Hinamatsuri' 2 | 'Mushishi' 1 | 'Macross Dynamite 7' 1 | 'Ashita no Joe (Movie)' 1 | 'Decorator' 1
title_english: 'Cyborg 009' 4 | 'Digimon: The Movie' 3 | 'Big Windup!' 3 | 'Fight! Kickers' 3 | 'The Heroic Legend of Arslan' 3
title_japanese: 'ゲゲゲの鬼太郎' 6 | 'うっかりペネロペ' 4 | '아기공룡 둘리' 4 | '宇宙の騎士テッカマンブレード' 4 | 'ぼのぼの' 4
title_synonyms: 'Minna no Uta' 189 | 'Irodorimidori' 23 | 'Sekai Meisaku Gekijou' 5 | 'Doutoku Meisaku Series' 5 | 'Backkom, Bernard' 4
image_url: 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1
type: 'TV' 4,271 | 'OVA' 3,612 | 'Movie' 2,624 | 'Special' 1,948 | 'ONA' 1,144
source: 'Unknown' 4,210 | 'Original' 3,368 | 'Manga' 3,120 | 'Visual novel' 888 | 'Game' 607
status: 'Finished Airing' 13,791 | 'Currently Airing' 351 | 'Not yet aired' 336
aired_string: 'Not available' 223 | '2005' 47 | '2004' 43 | 'Jul, 2018 to ?'

### Nhận xét cho lần nhìn qua đầu tiên

**Cách đọc.** Cell đầu đếm, trong mọi cột văn bản, các ô chỉ là một từ hay ký hiệu giống placeholder (*chính xác*), các văn bản dài hơn chứa từ đó (*một phần*) và các văn bản từ hai ký tự trở xuống (*ngắn*). Cell thứ hai in năm giá trị phổ biến nhất của mỗi cột. Nhiều kết quả là báo động giả, nên mỗi dòng dưới đây ghi rõ ta đã kiểm tra bằng mắt điều gì.

**1. Các từ placeholder chính xác của bảng anime: danh sách của ta đã đầy đủ.**

| Cột | Placeholder | Số ô |
|---|---|---|
| `source` | `Unknown` | 4,210 |
| `broadcast` | `Unknown` | 2,241 |
| `rating` | `None` | 544 |
| `duration` | `Unknown` | 405 |
| `aired_string` | `Not available` | 223 |
| `type` | `Unknown` | 30 |

Đây là cùng sáu cột và cùng số đếm với danh sách của phần 2. Việc quét **không tìm thấy từ nào khác** (không có `-`, `?`, `N/A`, `0` hay khoảng trắng) trong bất kỳ cột văn bản nào của bảng anime, và năm giá trị phổ biến nhất của mỗi cột cũng không có. Vì vậy danh sách `PLACEHOLDERS` của phần 2 là đủ cho bảng anime.

**2. Placeholder một phần: `broadcast` có thứ trong tuần nhưng không có giờ.** 78 ô có dạng `Sundays at Unknown` (15), `Tuesdays at Unknown` (13), `Fridays at Unknown` (13), `Mondays at Unknown` (11), `Saturdays at Unknown` (11), `Thursdays at Unknown` (10), và vài giá trị khác. Biết thứ nhưng không biết giờ. 65 ô khác ghi `Not scheduled once per week`, tức không có thứ cố định. Vậy giờ phát sóng không biết với 12,526 anime (86.5%): 10,207 ô rỗng, 2,241 ô `Unknown` và 78 ô "at Unknown".

**Các kết quả một phần còn lại là báo động giả.** Mẫu `tba` xuất hiện bên trong các từ bình thường (`GetBackers`, `SoftBank`, `Basketball`, `Football`), và `unknown` xuất hiện trong tựa thật (`The Unknown Girlfriend`) và tên bài hát (`Destination Unknown`). Hai bài opening có nghệ sĩ tên `Unknown`, không đáng kể.

**3. Danh sách rỗng và dictionary rỗng là placeholder mà việc quét từ không tìm ra.** Chúng là văn bản thật, nên `isna()` cũng không thấy.

| Cột | Giá trị | Số anime | Tỷ lệ trên 14,478 |
|---|---|---|---|
| `opening_theme` | `[]` | 9,784 | 67.6% |
| `ending_theme` | `[]` | 8,807 | 60.8% |
| `related` | `[]` | 4,515 | 31.2% |
| `aired` | `{'from': None, 'to': None}` | 1,691 | 11.7% |
| `opening_theme` | `['None']` | 7 | 0.05% |

- **Các cột theme rỗng với khoảng hai phần ba số anime**, nên không dùng được nguyên trạng. Tốt nhất chỉ còn một đặc trưng có/không ("có opening hay không").
- **`aired` không có ngày nào với 1,691 anime, nhưng `aired_string` chỉ là `Not available` với 223.** Vậy `aired_string` vẫn phải chứa thông tin cho khoảng 1,468 anime còn lại. Các giá trị phổ biến nhất của nó là một năm đơn lẻ (`2005` với 47 anime, `2004` với 43, `2000` với 40) và một tháng (`Jul, 2018 to ?` với 41 anime). Cách hiểu của ta, **được kiểm tra ở cell 1b bên dưới**, là `aired` bị rỗng khi ngày chỉ có một phần (năm hoặc tháng). Nếu đúng, phân tích theo năm phát hành phải đọc `aired_string` chứ không phải `aired`, vì `aired` làm mất 11.7% số anime.
- **Hệ quả cho notebook 02.** Ở đó ta đếm 147 anime có nhãn `Not yet aired` mà không có ngày bắt đầu, dựa trên `aired`. Một số trong đó có lẽ có ngày ở mức tháng như `Jul, 2018`, nằm sau thời điểm crawl và hợp với "chưa phát sóng". Điều này được kiểm tra ở cell 1b bên dưới (kiểm tra 4).

**4. Ngày bắt đầu có thể mang một ngày mặc định (giả thuyết).** Trong `aired`, ba trong số các ngày bắt đầu phổ biến nhất rơi vào ngày 11 của các tháng khác nhau: `2018-05-11` (57 anime: 39 cộng 18), `2018-07-11` (24) và `2018-10-11` (15). Lịch thật không có lý do ưu ái ngày 11. Trông giống một ngày tự động cho những ngày chỉ biết đến tháng. Điều này được kiểm tra ở cell 1b bên dưới (kiểm tra 2).

**5. Mã HTML còn sót trong tựa.** Một số văn bản `title_english` và `title_synonyms` chứa `&#039;` thay cho dấu nháy đơn (`Kuroko&#039;s Basketball`, `Kuroko&#039;s Basketball Special`). Ta chỉ thấy vì các tựa này tình cờ cũng khớp mẫu `tba`, nên chưa biết có bao nhiêu ô bị ảnh hưởng. Cell 1b bên dưới đếm chúng. Tựa phải được giải mã (`html.unescape`) trước khi so khớp hay đếm từ.

**6. Bảng user.**
- **`location`: 309 ô chỉ là placeholder** trong 156,799 ô có giá trị (0.2%): `Unknown` 61, `-` 42, `...` 28, `?` 23, `???` 20, `unknown` 18, và những giá trị khác không liệt kê. Thêm 78 ô chứa từ đó bên trong một văn bản dài hơn (`Parts Unknown` 8, `Unknown Location` 2, `unknown, Poland` 1). 26 từ của kiểm tra NA bị giấu bên dưới (`N/A`, `n/a`, `NA`, `None`) chỉ là một phần của 309 ô này, nên danh sách từ đó quá ngắn với `location`. Phần 2 không bao gồm bảng user, nên chúng cần danh sách riêng.
- **2,155 `location` có hai ký tự:** `UK` 668, `US` 109, `CA` 82, `NY` 70, `NJ` 57, `PA` 51. Chúng là giá trị thật (chữ viết tắt), nhưng `CA` có thể là Canada hoặc California, nên bước gom nhóm cần quy tắc.
- **15 username trông như placeholder** (`Unknown`, `unk`, `----`, `________`, `null`, ...), nhưng username là duy nhất và mỗi cái là một user thật. **Không username nào bị coi là thiếu.** 194 kết quả một phần là tên bình thường (`UnknownTitanGuy`).
- **`birth_date`: ngày 1 tháng 1 trông như ngày mặc định.** Năm ngày phổ biến nhất đều là 1 tháng 1: `1992-01-01` 630 user, `1993-01-01` 625, `1991-01-01` 599, `1994-01-01` 587 và `1990-01-01` 574. Nếu 168,741 ngày sinh phân bố đều trong năm, mỗi ngày chỉ có khoảng 18 user. Vậy 1 tháng 1 xuất hiện nhiều hơn khoảng 30 lần, hợp với "chỉ khai năm sinh" (giả thuyết, kiểm tra ở cell 1b bên dưới). Khi đó tuổi chỉ chính xác đến năm, phân tích theo tháng sinh là vô nghĩa, và năm sinh 1930 ở phần 6 có lẽ cũng là giá trị mặc định ấy.
- **`last_online`: một đỉnh đột biến, không phải placeholder.** Năm giá trị phổ biến nhất đều nằm vào 2013-09-13 giữa 12:52 và 13:38, với 15 đến 19 user trong một phút. Có chuyện gì đó xảy ra với nhiều tài khoản vào thời điểm đó. Ta không biết là gì.
- **`gender` có ba nhãn sạch** và không có placeholder.

**Điều này thay đổi gì.**
- Danh sách `PLACEHOLDERS` của phần 2 giữ nguyên cho bảng anime.
- Thêm vào các vấn đề của bước 4: danh sách rỗng và dictionary `None`, thứ trong tuần không có giờ, placeholder trong `location`, ngày sinh 1 tháng 1 và mã HTML (các dòng 17 đến 22 của bảng tóm tắt bên dưới).
- **Bốn kiểm tra** (mã HTML, ngày trong tháng của `aired`, tỷ lệ 1 tháng 1, `aired` so với `aired_string`) được thêm sau lần chạy này. Kết quả của chúng nằm ở cell 1b bên dưới.


### 1b. Bốn kiểm tra cho các phỏng đoán của phần 1a (MỚI, thêm sau lần chạy thứ hai)

**Vì sao.** Lần nhìn qua đầu tiên (1a) đưa ra bốn phỏng đoán mà ta ghi lại như giả thuyết. Mỗi phỏng đoán có thể biến thành một con số bằng hai bảng nhỏ đã nạp sẵn, nên ta kiểm tra ở đây trước khi dựa vào chúng.

| # | Phỏng đoán từ 1a | Cell đếm gì | Kết quả sẽ có nghĩa gì |
|---|---|---|---|
| 1 | Một số tựa vẫn chứa mã HTML như `&#039;` | Số ô chứa mã như `&#039;` hoặc `&amp;`, theo từng cột văn bản, và các mã phổ biến nhất | Nếu nhiều ô bị ảnh hưởng, tựa phải được giải mã (`html.unescape`) trước khi so khớp hay đếm từ |
| 2 | Ngày 11 có thể là ngày tự động trong ngày bắt đầu | Số ngày bắt đầu (`aired`) theo từng ngày trong tháng, từ 1 đến 31 | Nếu ngày 11 cao hơn hẳn các ngày khác, đó là ngày mặc định và các ngày ấy chỉ chính xác đến tháng |
| 3 | 1 tháng 1 có thể là ngày sinh mặc định | Tỷ lệ 1 tháng 1 trong tất cả ngày sinh, các cặp tháng-ngày phổ biến nhất, và tỷ lệ 1 tháng 1 theo vài năm sinh | Nếu 1 tháng 1 cao hơn hẳn 1/365 (0.27%), chỉ có thể tin năm sinh của những user đó |
| 4 | `aired` rỗng khi ngày chỉ có một phần, và `aired_string` vẫn giữ nó | Với anime có `aired` rỗng: `aired_string` chứa loại văn bản nào (ngày đầy đủ, tháng và năm, chỉ năm, rỗng). Tương tự với anime có status `Not yet aired` và `aired` rỗng (147 anime của notebook 02) | Nếu `aired_string` chứa năm hoặc tháng, năm phát hành phải đọc từ `aired_string`, và "chưa phát sóng mà không có ngày bắt đầu" không có nghĩa là "không có ngày nào" |

**Giới hạn.** Cell chỉ mô tả những gì có trong hai bảng nhỏ. Nó không cho biết vì sao có ngày mặc định, chỉ cho biết số đếm trông giống như có.


In [4]:
# NEW: four checks on the guesses of part 1a (they only read the two small tables)
import ast, html

# 1. HTML codes in the text columns
ENTITY = r"&#?\w+;"
print("1. Cells with an HTML code (for example &#039; or &amp;)")
for label, df in (("anime", anime), ("users", users)):
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        hit = s[s.str.contains(ENTITY, regex=True)]
        if len(hit):
            codes = hit.str.findall(ENTITY).explode().value_counts().head(5).to_dict()
            print(f"   {label}.{col}: {len(hit):,} of {len(s):,} cells ({len(hit) / len(s):.2%}) | codes: {codes}")
            print(f"      example: {hit.iloc[0][:60]!r} -> {html.unescape(hit.iloc[0])[:60]!r}")

# 2. Day of the month of the start date in `aired`
def aired_from(t):
    try:
        return pd.to_datetime(ast.literal_eval(t).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

start = pd.to_datetime(anime["aired"].map(aired_from), utc=True)
day = start.dt.day.dropna().astype(int)
per_day = day.value_counts().reindex(range(1, 32), fill_value=0)
print()
print(f"2. Day of the month of the start date: {len(day):,} start dates, mean per day {len(day) / 31:.0f}, median day count {per_day.median():.0f}")
print("   count per day:", per_day.to_dict())
print("   top 3 days:", per_day.sort_values(ascending=False).head(3).to_dict())
d11 = start[start.dt.day == 11]
print("   start dates on the 11th, most frequent year-months:", d11.dt.strftime("%Y-%m").value_counts().head(5).to_dict())

# 3. January 1 among the birth dates
bd = pd.to_datetime(users["birth_date"], errors="coerce").dropna()
month_day = bd.dt.strftime("%m-%d")
share_jan1 = (month_day == "01-01").mean()
print()
print(f"3. Birth dates: {len(bd):,} parsed. January 1: {(month_day == '01-01').sum():,} users = {share_jan1:.2%} (uniform days would give {1 / 365:.2%})")
print("   most frequent month-day values:", month_day.value_counts().head(5).to_dict())
by_year = bd[bd.dt.year.between(1985, 2000)]
jan1_by_year = (by_year.dt.strftime("%m-%d") == "01-01").groupby(by_year.dt.year).mean().round(3)
print("   share of January 1 per birth year, 1985-2000:", jan1_by_year.to_dict())

# 4. `aired` against `aired_string`
def kind(t):
    if pd.isna(t) or str(t).strip().lower() in ("", "not available"): return "empty"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", t): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", t): return "month and year"
    if re.search(r"\b\d{4}\b", t): return "year only"
    return "other"

string_kind = anime["aired_string"].map(kind)
no_start = start.isna()
print()
print(f"4. `aired` has no start date for {int(no_start.sum()):,} anime. What `aired_string` holds for them:")
print("  ", string_kind[no_start].value_counts().to_dict())
print("   other texts (first 8):", anime.loc[no_start & (string_kind == "other"), "aired_string"].value_counts().head(8).to_dict())
print("   `aired` has a start date but `aired_string` is empty:", int((~no_start & (string_kind == "empty")).sum()))
unaired = no_start & (anime["status"] == "Not yet aired")
print(f"   status `Not yet aired` and no start date in `aired`: {int(unaired.sum()):,}. What `aired_string` holds:")
print("  ", string_kind[unaired].value_counts().to_dict())
print("   most frequent texts:", anime.loc[unaired, "aired_string"].value_counts().head(6).to_dict())


1. Cells with an HTML code (for example &#039; or &amp;)
   anime.title: 202 of 14,478 cells (1.40%) | codes: {'&#039;': 213}
      example: 'Dae Jang Geum: Jang Geum&#039;s Dream' -> "Dae Jang Geum: Jang Geum's Dream"
   anime.title_english: 409 of 5,724 cells (7.15%) | codes: {'&#039;': 426}
      example: 'Sekai Ichi Hatsukoi - World&#039;s Greatest First Love 2' -> "Sekai Ichi Hatsukoi - World's Greatest First Love 2"
   anime.title_japanese: 76 of 14,443 cells (0.53%) | codes: {'&#039;': 79}
      example: '電脳戦隊ヴギィ&#039;ズ★エンジェル外伝 進め! スーパー★エンジェルス!' -> "電脳戦隊ヴギィ'ズ★エンジェル外伝 進め! スーパー★エンジェルス!"
   anime.title_synonyms: 739 of 8,937 cells (8.27%) | codes: {'&#039;': 898}
      example: 'Sekai-ichi Hatsukoi 2, Sekai&#039;ichi Hatsukoi 2' -> "Sekai-ichi Hatsukoi 2, Sekai'ichi Hatsukoi 2"
   anime.background: 316 of 1,057 cells (29.90%) | codes: {'&#039;': 378, '&rsquo;': 26, '&rdquo;': 11, '&mdash;': 11, '&ndash;': 8}
      example: 'Bleach adapts the first 54 volumes of Tite Kubo&#039;s man

### Nhận xét cho bốn kiểm tra

**Cách đọc.** Mỗi khối trong bốn khối output kiểm tra một phỏng đoán của phần 1a. Dưới đây, mỗi phỏng đoán nhận một kết luận cùng các con số đằng sau.

**1. Mã HTML: phổ biến, và hầu hết là `&#039;` (dấu nháy đơn).**

| Cột | Số ô có mã | Tỷ lệ trên các ô có giá trị |
|---|---|---|
| `anime.background` | 316 | 29.9% |
| `anime.title_synonyms` | 739 | 8.3% |
| `anime.title_english` | 409 | 7.1% |
| `anime.ending_theme` | 393 | 2.7% |
| `anime.opening_theme` | 303 | 2.1% |
| `anime.producer` | 168 | 2.0% |
| `anime.studio` | 130 | 1.5% |
| `anime.related` | 212 | 1.5% |
| `anime.title` | 202 | 1.4% |
| `anime.title_japanese` | 76 | 0.5% |
| `users.location` | 40 | 0.03% |

- Mã `&#039;` là mã chính (ví dụ `Brain&#039;s Base` thay cho `Brain's Base`). `&amp;` (ký tự `&`) xuất hiện trong `producer` (112) và `studio` (40). `background` còn có `&rsquo;`, `&rdquo;`, `&mdash;` và `&ndash;`, `related` có `&gt;`, và `users.location` có các chữ có dấu (`&eacute;`, `&atilde;`, `&ouml;`, ...).
- **Kết luận:** phỏng đoán đúng và lớn hơn ta nghĩ: 11 cột văn bản bị ảnh hưởng, và cùng một công ty hay tựa có thể tồn tại ở hai cách viết (`Brain's Base` và `Brain&#039;s Base`). Mọi cột văn bản phải được giải mã bằng `html.unescape` trước khi so khớp, gom nhóm theo studio hay producer, hoặc đếm từ.

**2. Ngày mặc định 11: không phải quy tắc chung, nhưng tồn tại với các ngày chỉ có một phần (giải thích ở phần 7).**

| Ngày trong tháng | Số ngày bắt đầu |
|---|---|
| 25 (cao nhất) | 808 |
| 21 | 678 |
| 5 | 623 |
| 1 | 612 |
| **11** | **457** |
| trung bình trên 31 ngày | 412 |
| ngày trung vị | 359 |
| 31 (thấp nhất, không phải tháng nào cũng có) | 159 |

- Trên tất cả 12,787 ngày bắt đầu, ngày 11 chỉ gấp 1.1 lần mức trung bình. Nó không phải một đỉnh riêng.
- Ngày 11 cao **chỉ ở vài tháng của năm 2018**: trong 457 ngày đó, 107 rơi vào `2018-05` (60), `2018-07` (27) và `2018-10` (20); mọi năm-tháng khác có tối đa 7. (Phần 1a đếm 57 cho `2018-05-11` từ văn bản chính xác; đếm theo ngày là 60.) Phần 7 (cell A5b) giải thích chúng: đó là những ngày được điền vào vì văn bản của `aired_string` chỉ viết một phần (ví dụ `Jul, 2018` được lưu cùng một ngày bịa ra).
- Hai đỉnh khác chưa giải thích được: ngày 25 (808) và ngày 21 (678). Chúng có thể là thói quen phát hành thật hoặc một giá trị mặc định khác; điều này chưa được kiểm tra.
- **Kết luận:** ngày 11 nói chung không phải ngày mặc định. Nhưng một ngày chỉ viết thành tháng và năm được lưu với một ngày điền vào (9 đến 12) và đôi khi một năm điền vào (phần 7, A5b), nên các ngày như vậy không đáng tin ở mức ngày. Các đỉnh ở ngày 21 và 25 vẫn để mở.

**3. 1 tháng 1 là ngày sinh mặc định: được ủng hộ.**
- **7,353 trong 168,741 ngày sinh đọc được là 1 tháng 1: 4.36%.** Nếu các ngày phân bố đều, tỷ lệ sẽ là 0.27% (khoảng 462 user). Vậy 1 tháng 1 **xuất hiện nhiều gấp 16 lần**, và khoảng 6,900 user có ngày 1 tháng 1 có lẽ không phải sinh nhật thật.
- Tỷ lệ này không giống nhau với mọi năm sinh: **6.5% với user sinh năm 1985 và 1.9% với user sinh năm 2000**, giảm gần như mỗi năm (1985: 6.5%, 1990: 4.7%, 1995: 3.5%, 2000: 1.9%). User lớn tuổi hơn thường được ghi 1 tháng 1 nhiều hơn.
- Các cặp tháng-ngày phổ biến tiếp theo là mùng 1 tháng 7 (619), tháng 4 (613), tháng 10 (595) và tháng 5 (589), khoảng 1.3 lần mức 462 kỳ vọng. Mức tăng nhỏ vào mùng 1 hàng tháng này chưa được kiểm tra sâu hơn.
- **Kết luận:** phỏng đoán khớp với dữ liệu. Ta không thể biết trong 7,353 ngày đó ngày nào là sinh nhật thật (khoảng 462 được kỳ vọng là thật). Vì vậy **năm sinh** là phần duy nhất đáng tin: dùng tuổi theo năm, không dùng tháng hay ngày sinh. Chưa kiểm tra: 842 user sinh năm 1930 (phần 6) có cũng là 1 tháng 1 hay không.

**4. `aired` so với `aired_string`: được xác nhận với anime không có ngày bắt đầu.**

| `aired_string` của 1,691 anime không có ngày bắt đầu trong `aired` | Số anime |
|---|---|
| Chỉ có năm (ví dụ `2005`) | 1,114 |
| Tháng và năm (ví dụ `Jul, 2018 to ?`) | 354 |
| Rỗng (`Not available`) | 223 |
| Ngày đầy đủ hoặc văn bản khác | 0 |

- Vậy `aired` rỗng khi ngày chỉ có một phần: **1,468 anime (86.8% trong 1,691, 10.1% trong tổng 14,478) có năm hoặc tháng trong `aired_string`** mà không có gì trong `aired`. Với 223 anime còn lại không có ngày nào. Ngoài ra, không anime nào có ngày bắt đầu trong `aired` mà `aired_string` lại rỗng.
- **Hệ quả:** năm phát hành phải đọc từ `aired_string` (một năm, hoặc tháng và năm). Chỉ đọc `aired` sẽ mất 1,468 anime. Chưa kiểm tra ở đây: một anime có `aired_string` chỉ có năm vẫn có thể có ngày bắt đầu trong `aired` hay không (ta chỉ kiểm tra một chiều). Phần 7 (A5b) cho thấy một số văn bản chỉ có một phần, như `Oct, 2007 to Oct, 2008`, **có** ngày bắt đầu trong `aired`, nhưng đó là ngày điền vào và sai.
- **147 anime có status `Not yet aired` và không có ngày bắt đầu (notebook 02):** `aired_string` là `Not available` với **113** anime (không có ngày nào), là một năm với 19 và là tháng và năm với 15. Các văn bản phổ biến nhất trong 34 anime có ngày một phần là `2018` (14), `2019` (5), `Mar, 2018` (5), `Oct, 2018` (2) và `Nov, 2018` (2). Vậy phỏng đoán đúng với 34 trong 147 anime: chúng có tháng hoặc năm, chủ yếu là 2018 hay 2019, hợp với "chưa phát sóng" tại thời điểm crawl (tháng 5/2018). Một ngoại lệ cần xem: `Mar, 2018` (5 anime) nằm trước thời điểm crawl, nên nhãn có thể đã cũ hoặc buổi ra mắt bị hoãn. Ta chưa kiểm tra. Kết luận của notebook 02 không bị yếu đi.

**Điều này thay đổi gì.**
- Các dòng 17 và 20 đến 22 của bảng tóm tắt được cập nhật bằng các số đếm ở trên.
- Cho bước 4: giải mã mã HTML trong mọi cột văn bản, đọc năm phát hành từ `aired_string`, coi 1 tháng 1 là ngày sinh không rõ, và không tin ngày hay năm của một ngày bắt đầu có văn bản chỉ có một phần (phần 7).
- **Các kiểm tra còn mở:** các đỉnh ở ngày 21 và 25, mùng 1 hàng tháng trong ngày sinh, 1 tháng 1 với 842 user sinh năm 1930, và chiều ngược lại của kiểm tra 4.


In [5]:
NA_TOKENS = {"null", "NULL", "NA", "N/A", "n/a", "NaN", "nan", "None", "#N/A", "#NA",
             "-NaN", "-nan", "<NA>", "1.#IND", "1.#QNAN", "-1.#IND", "-1.#QNAN", "#N/A N/A"}

for file_name in ["AnimeList.csv", "UserList.csv"]:
    text = pd.read_csv(paths[file_name], dtype=str, keep_default_na=False)
    hits = {col: int(text[col].isin(NA_TOKENS).sum()) for col in text.columns}
    hits = {col: count for col, count in hits.items() if count}
    print(file_name, "-> columns with hidden NA tokens:", hits or "none")
    for col in hits:
        print("   ", col, text.loc[text[col].isin(NA_TOKENS), col].value_counts().head(5).to_dict())
    del text

# Safe load: only the empty string is missing
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("\nanime:", anime.shape, "| users:", users.shape)
print("username NaN:", users.username.isna().sum(),
      "| username unique:", users.username.is_unique,
      "| distinct usernames:", f"{users.username.nunique():,}", "(description: 302,675)")
assert users.username.is_unique

AnimeList.csv -> columns with hidden NA tokens: {'rating': 544}
    rating {'None': 544}
UserList.csv -> columns with hidden NA tokens: {'username': 1, 'location': 26}
    username {'null': 1}
    location {'N/A': 15, 'n/a': 7, 'NA': 3, 'None': 1}

anime: (14478, 31) | users: (302675, 17)
username NaN: 0 | username unique: True | distinct usernames: 302,675 (description: 302,675)


### Nhận xét cho phần 1

**Ta đã làm gì.** Mọi ô của hai bảng nhỏ được đọc dưới dạng văn bản thô, và ta đếm các ô mà pandas sẽ biến thành giá trị thiếu (`null`, `NA`, `N/A`, `None`, ...). Sau đó cả hai bảng được nạp lại bằng cách nạp an toàn.

| File | Cột | Từ bị giấu | Số ô |
|---|---|---|---|
| `AnimeList.csv` | `rating` | `None` | 544 |
| `UserList.csv` | `username` | `null` | 1 |
| `UserList.csv` | `location` | `N/A` 15, `n/a` 7, `NA` 3, `None` 1 | 26 |

Không ô nào khác của hai bảng chứa các từ này.

- **`username`: cách sửa được xác nhận lần thứ hai.** Với cách nạp an toàn có 0 username thiếu và 302,675 username khác nhau, đúng như mô tả, và `assert` về tính duy nhất đã qua.
- **`rating = None` (544 anime).** Với cách nạp mặc định pandas sẽ biến chúng thành giá trị thiếu mà không báo cho ta. Chúng là một từ có nghĩa "không có xếp hạng độ tuổi". Có tính chúng là thiếu hay không là quyết định làm sạch của bước 4.
- **`location` (26 ô).** User tự gõ các từ này. `N/A`, `n/a` và `None` có nghĩa "không trả lời". `NA` (3 ô) có thể là cùng một kiểu không trả lời hoặc một chữ viết tắt, và ta không phân biệt được. 26 ô này chỉ là phần mà pandas giấu: lần nhìn qua đầu tiên (phần 1a) tìm thấy **309 ô placeholder trong `location`** (0.2% trong 156,799 ô có giá trị). Cách nạp an toàn giữ tất cả chúng dưới dạng văn bản.


## 2. Bảng anime

Các cột quan trọng cho các câu hỏi ứng viên: `type`, `source`, `episodes`, `duration`, `aired_string`, `genre`, `status`, `rating` (xếp hạng độ tuổi), `related` (phần tiếp theo, phần trước), và các cột phản hồi của khán giả `score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`.

Với mô hình dự đoán điểm **trước khi anime phát sóng**, các cột của khán giả (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) chỉ biết sau khi phát sóng, nên phải loại ra vì rò rỉ thông tin (leakage).


In [6]:
def missing_overview(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    }).sort_values("missing_pct", ascending=False)


print(anime.dtypes.value_counts().to_dict())
print(missing_overview(anime).head(12))

# Placeholder values that are not NaN but also carry no information
PLACEHOLDERS = {"", "Unknown", "Not available", "None", "?", "0000-00-00"}
text_cols = anime.select_dtypes(exclude="number").columns
placeholder_counts = pd.Series({c: int(anime[c].isin(PLACEHOLDERS).sum()) for c in text_cols})
print("\nplaceholder values per column:")
print(placeholder_counts[placeholder_counts > 0].sort_values(ascending=False))

print("\nduplicated anime_id:", anime.anime_id.duplicated().sum())

{dtype('O'): 22, dtype('int64'): 6, dtype('float64'): 2, dtype('bool'): 1}
                missing_count  missing_pct
background              13421        92.70
licensor                11105        76.70
premiered               10382        71.71
broadcast               10207        70.50
title_english            8754        60.46
producer                 6190        42.75
studio                   5934        40.99
title_synonyms           5541        38.27
rank                     1574        10.87
image_url                  96         0.66
genre                      64         0.44
title_japanese             35         0.24

placeholder values per column:
source          4210
broadcast       2241
rating           544
duration         405
aired_string     223
type              30
dtype: int64

duplicated anime_id: 0


In [7]:
# Numeric columns: are the ranges plausible?
num_cols = ["score", "scored_by", "rank", "popularity", "members", "favorites", "episodes"]
print(anime[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2))

# How reliable is the published score?
print("\nscore == 0:", (anime.score == 0).sum(),
      "| scored_by == 0:", (anime.scored_by == 0).sum(),
      "| both:", ((anime.score == 0) & (anime.scored_by == 0)).sum())
for k in (10, 50, 100, 1000):
    share = (anime.scored_by < k).mean()
    print(f"anime with scored_by < {k:>4}: {(anime.scored_by < k).sum():>6,} ({share:.1%})")
print("episodes == 0:", (anime.episodes == 0).sum())

              count      mean       std  min      1%      25%      50%  \
score       14478.0      6.14      1.46  0.0    0.00     5.55     6.37   
scored_by   14478.0  11460.03  43105.19  0.0    0.00    46.00   501.00   
rank        12904.0   6439.07   3720.23  0.0  120.06  3216.25  6441.50   
popularity  14478.0   7220.26   4170.08  0.0  136.77  3611.50  7225.50   
members     14478.0  22966.40  74981.36  0.0   29.00   245.00  1679.50   
favorites   14478.0    311.65   2615.55  0.0    0.00     0.00     2.00   
episodes    14478.0     11.31     43.44  0.0    0.00     1.00     1.00   

                 75%        99%        max  
score           7.06       8.46       10.0  
scored_by    3941.50  194311.41  1009477.0  
rank         9664.00   12742.97    12919.0  
popularity  10827.75   14300.23    14487.0  
members     10379.00  361159.94  1456378.0  
favorites      23.00    6780.55   106895.0  
episodes       12.00     104.00     1818.0  

score == 0: 354 | scored_by == 0: 308 | both: 

In [8]:
# Categorical columns
for col in ["type", "source", "status", "rating"]:
    print(f"\n{col}: {anime[col].nunique(dropna=False)} distinct values")
    print(anime[col].value_counts(dropna=False).head(12))


type: 7 distinct values
type
TV         4271
OVA        3612
Movie      2624
Special    1948
ONA        1144
Music       849
Unknown      30
Name: count, dtype: int64

source: 16 distinct values
source
Unknown         4210
Original        3368
Manga           3120
Visual novel     888
Game             607
Light novel      563
Other            408
Novel            359
Music            314
4-koma manga     224
Web manga        150
Picture book      96
Name: count, dtype: int64

status: 3 distinct values
status
Finished Airing     13791
Currently Airing      351
Not yet aired         336
Name: count, dtype: int64

rating: 7 distinct values
rating
PG-13 - Teens 13 or older         5020
G - All Ages                      4541
PG - Children                     1279
Rx - Hentai                       1219
R - 17+ (violence & profanity)     997
R+ - Mild Nudity                   878
None                               544
Name: count, dtype: int64


In [9]:
# Release year from aired_string (the first 4-digit number is the start year)
aired_year = pd.to_numeric(anime["aired_string"].str.extract(r"(\d{4})", expand=False), errors="coerce")
print("aired_string without a year:", aired_year.isna().sum(), f"({aired_year.isna().mean():.1%})")
print(aired_year.describe().round(1))
print(anime.loc[aired_year.isna(), "aired_string"].value_counts().head(5))
print("examples:", anime["aired_string"].dropna().head(5).tolist())

# Episode length in minutes from the text in `duration`
print("\n", anime["duration"].value_counts().head(8))
hours = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*hr")[0], errors="coerce").fillna(0)
minutes = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*min")[0], errors="coerce").fillna(0)
has_time = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
duration_min = (hours * 60 + minutes).where(has_time)
print("duration not parsable:", (~has_time).sum(), f"({(~has_time).mean():.1%})")
print(duration_min.describe(percentiles=[.01, .5, .99]).round(1))

aired_string without a year: 223 (1.5%)
count    14255.0
mean      2003.1
std         14.9
min       1917.0
25%       1997.0
50%       2008.0
75%       2014.0
max       2019.0
Name: aired_string, dtype: float64
aired_string
Not available    223
Name: count, dtype: int64
examples: ['Jan 13, 2012 to Mar 30, 2012', 'Apr 2, 2007 to Oct 1, 2007', 'Oct 4, 2008 to Sep 25, 2009', 'Aug 16, 2002 to May 23, 2003', 'Oct 6, 2012 to Mar 30, 2013']

 duration
24 min. per ep.    1513
25 min. per ep.     808
23 min. per ep.     797
30 min. per ep.     749
2 min.              476
Unknown             405
3 min.              371
4 min.              336
Name: count, dtype: int64
duration not parsable: 718 (5.0%)
count    13760.0
mean        26.8
std         25.5
min          1.0
1%           1.0
50%         24.0
99%        115.0
max        231.0
Name: 0, dtype: float64


In [10]:
# genre holds several values in one cell
genres = anime["genre"].dropna().str.split(", ")
print("anime without genre:", anime["genre"].isna().sum())
print("genres per anime:", genres.str.len().describe().round(2).to_dict())
print("distinct genres:", genres.explode().nunique())
print(genres.explode().value_counts().head(10))

# related: how are sequels stored, and how many anime have one?
print("\nexample:", anime["related"].dropna().head(2).tolist())
for relation in ["Sequel", "Prequel", "Side story", "Spin-off"]:
    share = anime["related"].str.contains(f"'{relation}'", na=False).mean()
    print(f"anime with a '{relation}' entry: {share:.1%}")

anime without genre: 64
genres per anime: {'count': 14414.0, 'mean': 2.91, 'std': 1.65, 'min': 1.0, '25%': 2.0, '50%': 3.0, '75%': 4.0, 'max': 13.0}
distinct genres: 43
genre
Comedy           5251
Action           3253
Fantasy          2703
Adventure        2582
Drama            2294
Sci-Fi           2272
Kids             2003
Shounen          1771
Romance          1634
Slice of Life    1532
Name: count, dtype: int64

example: ["{'Adaptation': [{'mal_id': 17207, 'type': 'manga', 'url': 'https://myanimelist.net/manga/17207/Inu_x_Boku_SS', 'title': 'Inu x Boku SS'}], 'Sequel': [{'mal_id': 13403, 'type': 'anime', 'url': 'https://myanimelist.net/anime/13403/Inu_x_Boku_SS_Special', 'title': 'Inu x Boku SS Special'}]}", "{'Adaptation': [{'mal_id': 759, 'type': 'manga', 'url': 'https://myanimelist.net/manga/759/Seto_no_Hanayome', 'title': 'Seto no Hanayome'}], 'Sequel': [{'mal_id': 4163, 'type': 'anime', 'url': 'https://myanimelist.net/anime/4163/Seto_no_Hanayome_OVA', 'title': 'Seto no Hanay

### Nhận xét cho phần 2 (bảng anime)

**Giá trị thiếu.** Các cột rỗng nhiều nhất giống như ở bước 2: `background` 92.7%, `licensor` 76.7%, `premiered` 71.7%, `broadcast` 70.5%, `title_english` 60.5%, `producer` 42.8%, `studio` 41.0%, `title_synonyms` 38.3%. `rank` thiếu 1,574 giá trị (10.9%), còn `genre` (64) và `title_japanese` (35) gần như đầy đủ. Không có `anime_id` nào lặp lại.

**Placeholder: những từ có nghĩa "không có giá trị" mà `isna()` không thấy.**

| Cột | Số ô chứa từ placeholder | Tỷ lệ trên 14,478 anime |
|---|---|---|
| `source` | 4,210 | 29.1% |
| `broadcast` | 2,241 | 15.5% |
| `rating` | 544 | 3.8% |
| `duration` | 405 | 2.8% |
| `aired_string` | 223 | 1.5% |
| `type` | 30 | 0.2% |

Toàn bộ 2,241 placeholder của `broadcast` là từ `Unknown` (phần 6). Với `broadcast`, placeholder cộng thêm vào các ô rỗng: 10,207 rỗng + 2,241 placeholder = 12,448 anime (86.0%) không có broadcast dùng được. Một cột rỗng hoặc là placeholder với 86% số dòng thì không phải một đặc trưng.

**Các con số: khoảng giá trị, và vì sao trung bình không phải giá trị điển hình.**

| Cột | Trung vị | Trung bình | Lớn nhất |
|---|---|---|---|
| `scored_by` | 501 | 11,460 | 1,009,477 |
| `members` | 1,680 | 22,966 | 1,456,378 |
| `favorites` | 2 | 312 | 106,895 |
| `episodes` | 1 | 11.3 | 1,818 |

Ở mọi dòng, trung bình cao hơn hẳn trung vị, nên một vài anime rất nổi tiếng kéo trung bình lên. Một nửa số anime có tối đa 1 tập (kể cả số 0), và một nửa có tối đa 501 người chấm. Với các cột này ta dùng trung vị hoặc thang log, không dùng trung bình.

**Score 0 nghĩa là "không có điểm".** 354 anime có `score = 0`. Trong đó 308 anime cũng có `scored_by = 0`, tức không ai chấm, và 0 là placeholder. 46 anime còn lại có `score = 0` dù `scored_by` lớn hơn 0. Phần 6 cho thấy mười anime có nhiều người chấm nhất đều là `Not yet aired`, với tối đa 15 người chấm: điểm không được công bố trước khi anime phát sóng. Dù thế nào, `score = 0` phải trở thành giá trị thiếu trước khi tính bất kỳ trung bình nào.

**Giá trị cần kiểm tra.**
- `rank` và `popularity` mỗi cột có **9 anime với giá trị 0**, dù rank bắt đầu từ 1 (đếm ở phần 6). Số 0 là placeholder.
- `episodes = 0` với **504 anime** (3.5%) nghĩa là "không biết số tập", không phải "không có tập nào". `episodes` bằng 1 với một nửa số anime (phim, special, video nhạc), và giá trị lớn nhất 1,818 là một series dài kỳ, hợp lý.

**Điểm `score` công bố đáng tin đến đâu?** Trung bình của vài phiếu thay đổi rất nhiều chỉ vì một phiếu. Năm phiếu 8 cho trung bình 8, thêm một phiếu 1 thì còn 6.83. Số anime có ít người chấm:

| Người chấm (`scored_by`) | Số anime dưới mức này | Tỷ lệ |
|---|---|---|
| 10 | 1,276 | 8.8% |
| 50 | 3,718 | 25.7% |
| 100 | 4,789 | 33.1% |
| 1,000 | 8,489 | 58.6% |

Cứ ba anime thì một có dưới 100 người chấm, nên `score` của nó nhiễu. Chỉ giữ anime có ít nhất 100 người chấm thì còn 9,689 trong 14,478. Nhóm phải chọn một ngưỡng cho mọi câu hỏi về chất lượng, và báo cáo ngưỡng đó loại bao nhiêu anime.

**Các biến phân loại.**
- `type` có 7 giá trị: TV 4,271, OVA 3,612, Movie 2,624, Special 1,948, ONA 1,144, Music 849 và `Unknown` 30.
- `source` có 16 giá trị, và `Unknown` (4,210) là giá trị phổ biến nhất.
- `rating` (xếp hạng độ tuổi) có 6 giá trị thật và `None` (544). `Rx - Hentai` có 1,219 anime (8.4%), điều này quan trọng với mọi câu hỏi về khán giả hay điểm số.
- `status` giống như ở bước 2: Finished Airing 13,791, Currently Airing 351, Not yet aired 336.

**Năm phát hành và độ dài một tập.**
- `aired_string` cho năm bắt đầu của 14,255 anime (98.5%). 223 anime còn lại chứa `Not available`. Các năm chạy từ 1917 đến 2019 với trung vị 2008 (một phần tư số anime bắt đầu sau 2014, một phần tư trước 1997). Không có năm nào không thể có. Năm 2019 thuộc về các anime chưa phát sóng tại thời điểm crawl.
- `duration` đổi được sang phút cho 13,760 anime. 718 anime (5.0%) code của ta không đọc được: 405 là `Unknown` (thiếu thật) và 313 là độ dài viết bằng giây, như `30 sec.` (phần 6), mà code của ta không đọc. Chỉ 405 là thiếu. Trung vị là 24 phút, và 99% có tối đa 115 phút (phim giữ tổng độ dài).

**Genre và `related`.**
- `genre` chứa nhiều giá trị trong một ô. 14,414 anime có genre mang trung bình 2.9 genre (từ 1 đến 13), với 43 genre khác nhau. Phổ biến nhất là Comedy (5,251 anime), Action (3,253) và Fantasy (2,703). Để đếm theo genre phải tách ô thành một dòng cho mỗi genre.
- `related` đọc được (là văn bản của một dictionary). Khoảng 16.8% số anime (quanh 2,430) có mục `Sequel` và cùng tỷ lệ đó có mục `Prequel`, hợp với các liên kết phản chiếu ta đã đo ở bước 5a. 7.3% có `Side story` và 2.5% có `Spin-off`. Vậy một câu hỏi về phần tiếp theo dựa trên khoảng 2,400 anime có phần tiếp theo.

**Điều này có nghĩa gì với mô hình dự đoán điểm trước khi anime phát sóng.** Các cột của khán giả (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) chỉ biết sau khi phát sóng, nên bị loại (leakage). Các cột còn lại là `type`, `source`, `episodes`, `duration`, `genre`, `rating`, `studio`, `producer`, năm bắt đầu và `related`. Vấn đề của chúng là `source = Unknown` với 29%, `studio` và `producer` thiếu khoảng 41% và 43%, `episodes = 0` với 3.5%, và `rating = None` với 3.8%.


## 3. Bảng user


In [11]:
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
stat_cols = ["user_days_spent_watching", "stats_mean_score", "stats_rewatched", "stats_episodes"]

print(users.dtypes.value_counts().to_dict())
print(missing_overview(users).head(8))
print("access_rank non-missing:", users["access_rank"].notna().sum())

print("\ngender:")
print(users["gender"].value_counts(dropna=False))
print("\ndistinct locations:", users["location"].nunique(),
      "| missing:", users["location"].isna().sum())
print(users["location"].value_counts().head(10))

{dtype('O'): 6, dtype('int64'): 6, dtype('float64'): 5}
                  missing_count  missing_pct
access_rank              302675       100.00
location                 145876        48.20
birth_date               133926        44.25
gender                    84875        28.04
stats_episodes              129         0.04
join_date                   129         0.04
stats_rewatched             129         0.04
stats_mean_score            129         0.04
access_rank non-missing: 0

gender:
gender
Male          135181
NaN            84875
Female         81374
Non-Binary      1245
Name: count, dtype: int64

distinct locations: 55283 | missing: 145876
location
Poland         2425
Germany        1947
Canada         1821
California     1510
Brazil         1459
France         1135
Philippines    1077
Australia      1032
Sweden         1013
Finland         898
Name: count, dtype: int64


In [12]:
# Dates: placeholders and implausible values
for col in ["birth_date", "join_date", "last_online"]:
    parsed = pd.to_datetime(users[col], errors="coerce")
    print(f"{col:12s} unparsable/missing: {parsed.isna().sum():>7,}",
          "| '0000-00-00':", int((users[col] == "0000-00-00").sum()),
          "| earliest:", parsed.min(), "| latest:", parsed.max())
    users[col + "_dt"] = parsed

age_2018 = (pd.Timestamp("2018-01-01") - users["birth_date_dt"]).dt.days / 365.25
print("\nage at 2018-01-01:")
print(age_2018.describe(percentiles=[.01, .05, .5, .95, .99]).round(1))
print("age < 8:", int((age_2018 < 8).sum()), "| age > 80:", int((age_2018 > 80).sum()))
print("last_online before 2000:", int((users["last_online_dt"].dt.year < 2000).sum()),
      "| joined after last_online:", int((users["join_date_dt"] > users["last_online_dt"]).sum()))

# Aggregates reported per user
print(users[stat_cols].describe(percentiles=[.01, .5, .99, .999]).T.round(2))
print("stats_mean_score == 0:", int((users["stats_mean_score"] == 0).sum()))
print("users declaring no anime at all:", int((users[count_cols].sum(axis=1) == 0).sum()))

birth_date   unparsable/missing: 133,934 | '0000-00-00': 0 | earliest: 1800-01-01 00:00:00 | latest: 2137-01-01 00:00:00
join_date    unparsable/missing:     129 | '0000-00-00': 0 | earliest: 2004-11-05 00:00:00 | latest: 2018-05-24 00:00:00
last_online  unparsable/missing:     129 | '0000-00-00': 0 | earliest: 1900-01-01 00:13:00 | latest: 2018-05-25 12:53:00

age at 2018-01-01:
count    168741.0
mean         24.9
std           7.0
min        -119.0
1%           14.7
5%           17.6
50%          24.2
95%          33.0
99%          44.4
max         218.0
Name: birth_date_dt, dtype: float64
age < 8: 769 | age > 80: 955
last_online before 2000: 89283 | joined after last_online: 89283
                             count     mean        std  min   1%      50%  \
user_days_spent_watching  302675.0    69.08    4741.02  0.0  0.0    39.58   
stats_mean_score          302546.0     7.12       2.57  0.0  0.0     7.80   
stats_rewatched           302546.0    10.38      44.24  0.0  0.0     0.00   

### Nhận xét cho phần 3 (bảng user)

**Giá trị thiếu.** `access_rank` rỗng với cả 302,675 user, nên ta bỏ nó. `location` thiếu 48.2%, `birth_date` thiếu 44.3% và `gender` thiếu 28.0%. `join_date`, `last_online` và ba cột `stats_*` mỗi cột thiếu 129 giá trị (cùng 129 user, xem phần 4 của notebook 02).

**Gender.** Trong 217,800 user có khai gender: Male 135,181 (62.1%), Female 81,374 (37.4%) và Non-Binary 1,245 (0.6%). Các nhãn sạch. Non-Binary là nhóm nhỏ, nên một phép so sánh có nhóm này sẽ có độ bất định lớn.

**Location.** 55,283 văn bản khác nhau. Phổ biến nhất là Poland (2,425), Germany (1,947), Canada (1,821), California (1,510) và Brazil (1,459). Các quốc gia và một tiểu bang của Mỹ nằm cạnh nhau, nên phân tích theo quốc gia cần một bước gom nhóm trước.

**Ngày.**

| Cột | Sớm nhất | Muộn nhất | Ý nghĩa |
|---|---|---|---|
| `join_date` | 2004-11-05 | 2018-05-24 | Hợp lý (MyAnimeList bắt đầu năm 2004) |
| `last_online` | 1900-01-01 | 2018-05-25 12:53 | Năm 1900 là không thể, xem bên dưới |
| `birth_date` | 1800-01-01 | 2137-01-01 | Có năm không thể ở cả hai đầu |

- **`last_online` chứa năm 1900 với nhiều user.** 89,283 user (29.5%) có `last_online` trước năm 2000, và chính 89,283 user đó cũng "tham gia sau khi online lần cuối". Phần 6 cho thấy **cả 89,283 đều đúng vào năm 1900** (ví dụ 1900-05-13 02:47), trong khi các user khác trải từ 2007 đến 2018. Vậy đây là một năm mặc định, không phải ngày thật. Năm có lẽ bị mất khỏi nguồn, nên tháng, ngày và giờ có thể đúng còn năm thì không. Năm thật không biết (các ví dụ ta thấy là từ tháng 3 đến tháng 5). Tác giả nói bản đã làm sạch sửa các giá trị này. Với 29.5% số user, `last_online` không dùng được làm ngày.
- **`birth_date`.** 8 văn bản không đọc được (133,934 không đọc được so với 133,926 rỗng). Chúng là ngày giả với các năm 1, 1337, 6969 và 9001 (phần 6). 168,741 user có ngày dùng được. Tuổi tại 2018-01-01 trung bình là 24.9 và trung vị là 24.2. 98% user ở giữa có tuổi từ 14.7 đến 44.4, nên phần lớn là hợp lý. Nhưng **769 user dưới 8 tuổi và 955 user trên 80 tuổi** (1,724 user, 1.0% số user có ngày), với các giá trị cực đoan -119 và 218 tuổi. Đó là ngày sai hoặc ngày đùa. Phần 6 bổ sung rằng 769 user nhỏ tuổi trải đều trên các năm sinh 2010 đến 2018, trong khi **842 trong 955 user già sinh năm 1930**, trông như một giá trị mặc định (giả thuyết: năm đầu tiên của một danh sách thả xuống). Mọi phân tích tuổi phải cắt khoảng tuổi.
- **Ngày crawl, góc nhìn thứ ba.** `last_online` muộn nhất là 2018-05-25 12:53, `join_date` muộn nhất là 2018-05-24, và trước đó ta tìm thấy 2018-05-22 cho `my_last_updated` và 2018-05-25 cho các ngày bắt đầu của anime. Cả bốn đều khớp: dữ liệu được đọc quanh 22 đến 25 tháng 5 năm 2018.

**Các con số MyAnimeList báo cho mỗi user.**

| Cột | Trung vị | Phân vị 99 | Lớn nhất | Vấn đề |
|---|---|---|---|---|
| `user_days_spent_watching` | 39.6 | 304 | **2,601,351** | Giá trị lớn nhất khoảng 7,100 năm |
| `stats_mean_score` | 7.80 | 10.00 | 10.00 | 30,922 user có giá trị 0 |
| `stats_rewatched` | 0 | 149 | 9,404 | Ý nghĩa chưa rõ (bước 5) |
| `stats_episodes` | 2,384 | 19,893 | **243,794,900** | Giá trị lớn nhất là không thể |

- **`stats_mean_score = 0` với 30,922 user (10.2% trong 302,546 user có giá trị).** Cũng như `my_score`, 0 ở đây nghĩa là "chưa cho điểm", không phải điểm trung bình 0, nên phải trở thành giá trị thiếu.
- **Giá trị lớn nhất không thể có.** Trung bình của `stats_episodes` (4,846) cao hơn hẳn trung vị (2,384) vì các giá trị như 243,794,900 tập, tương đương khoảng 11,000 năm xem liên tục với 24 phút mỗi tập. Giá trị 10,989,789 mà ta gặp ở bước 5a là lớn nhất trong các user được so sánh ở đó; trên toàn bộ user, giá trị lớn nhất là 243,794,900. Những giá trị này cần được chặn trần hoặc loại bỏ. Phần 6 đếm số user vượt qua vài mức chặn (204 user trên 100,000 tập, 20 user trên 1,000,000, và 42 user trên 3,650 ngày) và cho thấy các giá trị lớn nhất đến từ số tập vô lý.
- **19,628 user không khai anime nào.** Phần 4 giải thích mối liên hệ với 19,630 user không có dòng trong bảng list.


## 4. Số đếm theo từng user có khớp với bảng list không?

`UserList` báo mỗi user có bao nhiêu anime ở mỗi trạng thái. `UserAnimeList` chứa các dòng thật. Nếu cả hai đến từ cùng một lần crawl, số đếm phải khớp. Việc này cũng giải quyết 8 user ở bước 1 khai có anime nhưng không có dòng nào.


In [13]:
if RUN_LISTS:
    lists = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "my_status"],
        dtype={"username": "category", "my_status": "int8"}, **na_kwargs,
    )
    print(f"list rows: {len(lists):,} | username NaN: {lists.username.isna().sum()}")

    per_status = lists.groupby(["username", "my_status"], observed=True).size().unstack(fill_value=0)
    print("usernames in lists but not in users:", len(set(per_status.index) - set(users.username)))

    status_to_col = {1: "user_watching", 2: "user_completed", 3: "user_onhold",
                     4: "user_dropped", 6: "user_plantowatch"}
    actual = per_status.reindex(columns=list(status_to_col)).fillna(0).astype(int)
    actual.columns = [c + "_actual" for c in status_to_col.values()]
    actual.index = actual.index.astype(object)
    declared = users.set_index("username")[count_cols].add_suffix("_declared")
    merged = declared.join(actual, how="left").fillna(0)
    print("merge rows:", len(merged), "| users:", len(users))

    for col in count_cols:
        diff = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} exact match: {(diff == 0).mean():6.1%} | median diff: {diff.median():.0f}"
              f" | mean abs diff: {diff.abs().mean():.2f}")

    declared_total = merged[[c + "_declared" for c in count_cols]].sum(axis=1)
    actual_total = merged[[c + "_actual" for c in count_cols]].sum(axis=1)
    print("\ndeclare 0 but have rows:", int(((declared_total == 0) & (actual_total > 0)).sum()),
          "| declare > 0 but have no rows:", int(((declared_total > 0) & (actual_total == 0)).sum()))
else:
    print("Skipped (RUN_LISTS = False)")

list rows: 80,076,112 | username NaN: 0
usernames in lists but not in users: 0
merge rows: 302675 | users: 302675
user_watching      exact match:  98.2% | median diff: 0 | mean abs diff: 0.03
user_completed     exact match:  98.7% | median diff: 0 | mean abs diff: 0.07
user_onhold        exact match:  99.9% | median diff: 0 | mean abs diff: 0.01
user_dropped       exact match:  99.9% | median diff: 0 | mean abs diff: 0.01
user_plantowatch   exact match:  99.4% | median diff: 0 | mean abs diff: 0.04

declare 0 but have rows: 6 | declare > 0 but have no rows: 8


### Nhận xét cho phần 4 (số đếm khai báo so với bảng list)

**Ta đã làm gì.** `UserList` cho biết mỗi user có bao nhiêu anime ở mỗi trạng thái. Ta đếm số dòng thật của `UserAnimeList` theo từng user và trạng thái, rồi so sánh hai số. Mọi user của bảng user đều nằm trong phép so sánh (302,675 dòng), và 0 username của bảng list bị thiếu trong bảng user, nên phép join trên `username` không có dòng mồ côi ở cả hai phía.

| Trạng thái | User có số khai báo và số thật bằng nhau | Chênh lệch tuyệt đối trung bình |
|---|---|---|
| watching | 98.2% | 0.03 |
| completed | 98.7% | 0.07 |
| on hold | 99.9% | 0.01 |
| dropped | 99.9% | 0.01 |
| plan to watch | 99.4% | 0.04 |

Chênh lệch trung vị là 0 ở mọi trạng thái.

- **Hai bảng khớp nhau với ít nhất 98.2% số user ở mỗi trạng thái.** Số đếm trong profile và các dòng của list đến từ cùng một lần crawl và mô tả cùng các list. Bảng list gần như đầy đủ với những user mà nó có.
- **Sự không khớp nhỏ và lớn nhất ở `watching` (1.8% số user, khoảng 5,400) và `completed` (1.3%, khoảng 3,900).** Đây là các trạng thái thay đổi thường xuyên nhất, nên một lý do hợp lý là user đã sửa list giữa lúc profile được đọc và lúc list được đọc. Đây là một giả thuyết. Output không cho thấy chiều của các chênh lệch.
- **8 user khai có anime nhưng không có dòng nào.** Đây là cùng nhóm như ở bước 1, nay được xác nhận bằng một phương pháp thứ hai.
- **6 user khai 0 anime nhưng có dòng.** Đây là trường hợp ngược lại và là mới. Không biết lý do, và 6 user là quá ít để quan trọng.
- **Chiều (phần 6).** Với `completed`, list có ít dòng hơn profile trong 91% số trường hợp không khớp, và với `plan to watch` list có nhiều dòng hơn trong 79%. Điều này hợp với việc list được đọc sớm hơn profile. 6 user khai 0 anime nhưng có dòng có list dài (294 đến 2,057 dòng) và chưa được giải thích.
- **Hệ quả.** Số đếm của `UserList` có thể dùng làm bản tóm tắt của list, với một biên sai số nhỏ. Ta không cần coi chúng là không đáng tin.


## 5. Những user không có dòng nào trong bảng list: bảng list có thiếu họ không?

Ở bước 1 ta tìm thấy **19,630 user** có trong `UserList` nhưng không sở hữu dòng nào trong `UserAnimeList`. Với 19,622 trong số đó, profile khai 0 anime, nên một list rỗng là lời giải thích tự nhiên. Nhưng số đếm khai báo chỉ là một cột. Nếu những user này cũng có **giá trị hoạt động khác** (số ngày xem, số tập đã xem, điểm trung bình), list của họ có thể tồn tại trên MyAnimeList và chỉ đơn giản là thiếu trong bảng list của ta.

| Ta thấy gì ở những user này | Điều đó có nghĩa gì |
|---|---|
| `user_days_spent_watching`, `stats_episodes` và `stats_mean_score` đều bằng 0 | List thật sự rỗng. Không thiếu gì cả |
| Nhiều user có giá trị dương ở đó | Họ đã xem và chấm anime, nên list của họ **bị thiếu trong bảng list** (không được crawl, ở chế độ riêng tư, hoặc bị mất) |
| Các trường profile (gender, ngày sinh, location) được điền thường xuyên như với các user khác | Họ là profile hoạt động bình thường. Nếu các trường đó phần lớn rỗng, họ có lẽ là tài khoản không hoạt động |


In [14]:
if RUN_LISTS:
    users["has_rows"] = users["username"].isin(actual.index)
    no_rows = users[~users["has_rows"]]
    with_rows = users[users["has_rows"]]
    declared_size = no_rows[count_cols].sum(axis=1)
    print(f"users without a row in the list table: {len(no_rows):,} | with rows: {len(with_rows):,}")
    print(f"  declared list size 0: {int((declared_size == 0).sum()):,} | declared list size > 0: {int((declared_size > 0).sum()):,}")

    print("\nactivity values of the users without rows (an empty account would show 0 everywhere):")
    for col in ["user_days_spent_watching", "stats_episodes", "stats_rewatched", "stats_mean_score"]:
        s = no_rows[col]
        print(f"  {col:26s} missing: {int(s.isna().sum()):>6,} | zero: {int((s == 0).sum()):>6,} | positive: {int((s > 0).sum()):>6,}"
              f" | max: {s.max():,.1f}")
    active = ((no_rows["user_days_spent_watching"] > 0) | (no_rows["stats_episodes"] > 0) | (no_rows["stats_mean_score"] > 0))
    print(f"\nusers without rows that show activity in at least one of the three columns: {int(active.sum()):,}")

    print("\nprofile fields filled (share of users), compared with the users that have rows:")
    fields = ["gender", "birth_date", "location", "join_date", "last_online"]
    print(pd.DataFrame({"no_rows": no_rows[fields].notna().mean(),
                        "with_rows": with_rows[fields].notna().mean()}).round(3).to_string())

    print("\nusers without rows that declare anime (the unexplained group):")
    cols_show = ["username"] + count_cols + ["user_days_spent_watching", "stats_episodes", "stats_mean_score"]
    print(no_rows.loc[declared_size > 0, cols_show].sort_values("user_completed", ascending=False).head(15).to_string(index=False))
else:
    print("Skipped (RUN_LISTS = False)")

users without a row in the list table: 19,630 | with rows: 283,045
  declared list size 0: 19,622 | declared list size > 0: 8

activity values of the users without rows (an empty account would show 0 everywhere):
  user_days_spent_watching   missing:      0 | zero: 19,624 | positive:      6 | max: 260.7
  stats_episodes             missing:      2 | zero: 19,593 | positive:     35 | max: 30,300.0
  stats_rewatched            missing:      2 | zero: 19,624 | positive:      4 | max: 112.0
  stats_mean_score           missing:      2 | zero: 19,593 | positive:     35 | max: 10.0

users without rows that show activity in at least one of the three columns: 47

profile fields filled (share of users), compared with the users that have rows:
             no_rows  with_rows
gender         0.311      0.748
birth_date     0.263      0.578
location       0.221      0.539
join_date      1.000      1.000
last_online    1.000      1.000

users without rows that declare anime (the unexplained group):


### Nhận xét cho phần 5 (những user không có dòng nào trong bảng list)

**Câu hỏi.** Ở bước 1 ta tìm thấy 19,630 user không có dòng nào trong bảng list (283,045 user có dòng). Bảng list có thiếu list của họ không, hay họ là tài khoản rỗng? Ta nhìn các giá trị khác trong profile của họ: nếu họ hoạt động, họ sẽ có số ngày xem, số tập hoặc điểm trung bình lớn hơn 0.

| Giá trị của 19,630 user không có dòng | Bằng 0 | Lớn hơn 0 | Thiếu | Lớn nhất |
|---|---|---|---|---|
| `user_days_spent_watching` | 19,624 | 6 | 0 | 260.7 |
| `stats_episodes` | 19,593 | 35 | 2 | 30,300 |
| `stats_mean_score` | 19,593 | 35 | 2 | 10 |
| `stats_rewatched` | 19,624 | 4 | 2 | 112 |

- **Các user là tài khoản rỗng, với vài ngoại lệ.** 19,622 trong số họ khai kích thước list là 0, và **chỉ 47 user (0.24%) có hoạt động** ở ít nhất một trong ba cột. Với 19,583 user còn lại (99.76%) ba cột bằng 0 (hoặc thiếu với 2 user), đúng như một tài khoản không có list trông ra sao. Với họ không thiếu gì.
- **Profile của họ được điền ít hơn.** Tỷ lệ user có điền trường đó:

| Trường | User không có dòng | User có dòng |
|---|---|---|
| `gender` | 31.1% | 74.8% |
| `birth_date` | 26.3% | 57.8% |
| `location` | 22.1% | 53.9% |
| `join_date`, `last_online` | 100% | 100% |

  User không có dòng điền gender, ngày sinh và location ít hơn khoảng 2.2 đến 2.4 lần. Điều này hợp với những tài khoản được tạo ra rồi không bao giờ dùng. (2 user thiếu `join_date` là các user có profile rỗng của notebook 02.)
- **8 user khai có anime là nhóm còn mở.** Sáu người trong số họ có số ngày xem lớn hơn 0 (từ 0.22 đến 260.7 ngày; `Useless_Goddess` khai 1,043 anime completed), nhưng cả 8 đều có `stats_episodes = 0` và `stats_mean_score = 0`. Vậy profile của họ có số đếm nhưng không có các tóm tắt về tập và điểm. Một người tên là `deactivated-180120`, nhưng phần 6 cho thấy cái tên không giải thích được gì: 10 username bắt đầu bằng `deactivated`, và 6 trong số đó vẫn sở hữu dòng trong list. Vậy ta không giải thích được bất kỳ ai trong 8 người này (list riêng tư, hoặc list không được crawl).
- **Quyết định cho bước 4 (nhóm quyết định):** 19,583 tài khoản rỗng có thể bỏ khỏi mọi phân tích list. Một phép inner merge của hai bảng tự làm việc này, và không loại dòng nào của bảng list. 47 user có hoạt động và 8 user có list khai báo được báo cáo là những user mà list có thể bị thiếu.


## 6. Kiểm tra cho rõ (MỚI, thêm sau lần chạy đầu)

Phần 1 đến 5 đã chạy một lần. Một số câu trong phần nhận xét của chúng dựa trên các con số mà lần chạy đầu chưa in, và một số giả thuyết chưa từng được kiểm tra. Phần này in chúng ra. Nó không thay đổi gì trong dữ liệu, và dùng lại các đối tượng đã tạo ở trên (`anime`, `users`, `age_2018`, `merged`, ...), nên hãy chạy notebook từ đầu.

| Kiểm tra | Nó giải quyết điều gì |
|---|---|
| A. Giá trị anime | Có bao nhiêu `rank` và `popularity` bằng 0? Những từ nào điền vào `broadcast`? 313 văn bản `duration` không đọc được là gì? Những anime nào có `score = 0` nhưng có người chấm? |
| B. Ngày và tuổi của user | `last_online` trước năm 2000 có luôn là năm 1900 không (giả thuyết của ta)? Những văn bản `birth_date` nào không đọc được? Những năm sinh nào cho ra tuổi không thể có? |
| C. Giá trị cực đoan | Có bao nhiêu user vượt qua vài mức chặn của `stats_episodes` và `user_days_spent_watching`, và các giá trị lớn nhất trông thế nào? Việc này giúp chọn mức chặn ở bước 4. |
| D. Username `deactivated` | Có bao nhiêu username bắt đầu bằng `deactivated`, và họ có sở hữu dòng trong list không? (Một trong 8 user khai có anime mà không có dòng là `deactivated-180120`.) |
| E. Số khai báo so với số thật | Các số đếm lệch nhau theo chiều nào, và 6 user khai 0 anime nhưng có dòng là ai? |


In [15]:
# NEW A: anime values that the first run did not print
print("rank == 0:", int((anime["rank"] == 0).sum()), "| rank missing:", int(anime["rank"].isna().sum()),
      "| popularity == 0:", int((anime["popularity"] == 0).sum()))

print("\nbroadcast: the words that count as placeholders")
print(anime.loc[anime["broadcast"].isin(PLACEHOLDERS), "broadcast"].value_counts().to_string())

has_time_text = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
print("\nduration: the texts that cannot be parsed into minutes")
print(anime.loc[~has_time_text, "duration"].value_counts(dropna=False).head(10).to_string())

odd_score = (anime["score"] == 0) & (anime["scored_by"] > 0)
print("\nscore == 0 although scored_by > 0:", int(odd_score.sum()))
print(anime.loc[odd_score, ["anime_id", "title", "type", "status", "scored_by", "members"]]
      .sort_values("scored_by", ascending=False).head(10).to_string(index=False))


rank == 0: 9 | rank missing: 1574 | popularity == 0: 9

broadcast: the words that count as placeholders
broadcast
Unknown    2241

duration: the texts that cannot be parsed into minutes
duration
Unknown            405
30 sec.             54
30 sec. per ep.     42
15 sec.             30
15 sec. per ep.     17
40 sec. per ep.      8
45 sec.              7
39 sec. per ep.      6
52 sec.              5
16 sec.              5

score == 0 although scored_by > 0: 46
 anime_id                                                   title    type        status  scored_by  members
    37799                               Tokyo Ghoul:re 2nd Season      TV Not yet aired         15    28258
    37675                                            Overlord III      TV Not yet aired         12    60107
    35760                             Shingeki no Kyojin Season 3      TV Not yet aired         11   170892
    34134                                         One Punch Man 2      TV Not yet aired         10   234

In [16]:
# NEW B: user dates and ages
last_year = users["last_online_dt"].dt.year
print("last_online before 2000, by year:", last_year[last_year < 2000].value_counts().sort_index().to_dict())
print("last_online year counts from 2000 on:", last_year[last_year >= 2000].value_counts().sort_index().to_dict())

unparsable = users["birth_date"].notna() & users["birth_date_dt"].isna()
print("\nbirth_date texts that cannot be parsed:", users.loc[unparsable, "birth_date"].tolist())

birth_year = users["birth_date_dt"].dt.year
print("birth years of the users under 8 at 2018-01-01:", birth_year[age_2018 < 8].value_counts().head(8).to_dict())
print("birth years of the users over 80 at 2018-01-01:", birth_year[age_2018 > 80].value_counts().head(8).to_dict())


last_online before 2000, by year: {1900.0: 89283}
last_online year counts from 2000 on: {2007.0: 638, 2008.0: 5799, 2009.0: 11576, 2010.0: 14067, 2011.0: 12666, 2012.0: 12653, 2013.0: 17328, 2014.0: 15868, 2015.0: 18420, 2016.0: 24231, 2017.0: 40663, 2018.0: 39354}

birth_date texts that cannot be parsed: ['1-06-27', '1-04-22', '1-01-01', '1337-01-01', '1-01-01', '1337-01-01', '9001-12-31', '6969-01-01']
birth years of the users under 8 at 2018-01-01: {2012.0: 91, 2013.0: 87, 2011.0: 86, 2018.0: 82, 2014.0: 82, 2015.0: 79, 2010.0: 73, 2016.0: 71}
birth years of the users over 80 at 2018-01-01: {1930.0: 842, 1931.0: 41, 1934.0: 15, 1933.0: 14, 1937.0: 13, 1932.0: 13, 1935.0: 8, 1936.0: 8}


In [17]:
# NEW C: how many users lie above several caps, and what do the largest look like?
caps = {"stats_episodes": [10_000, 50_000, 100_000, 1_000_000],
        "user_days_spent_watching": [365, 1_000, 3_650, 36_500]}
for col, values in caps.items():
    print(col, {f"> {v:,}": int((users[col] > v).sum()) for v in values})

print()
show = ["username", "stats_episodes", "user_days_spent_watching", "user_completed", "user_plantowatch"]
print(users.nlargest(8, "stats_episodes")[show].to_string(index=False))


stats_episodes {'> 10,000': 19176, '> 50,000': 602, '> 100,000': 204, '> 1,000,000': 20}
user_days_spent_watching {'> 365': 1673, '> 1,000': 192, '> 3,650': 42, '> 36,500': 3}

       username  stats_episodes  user_days_spent_watching  user_completed  user_plantowatch
        warrose     243794881.0                2601351.02            2034                 7
      mmmapples      18218986.0                  23640.42               0                 0
        Kaiyuso      16791524.0                   2278.30             380               479
darkraigiratina      16776960.0                   8555.96               0                 2
 Kitayama_Naoki      10989789.0                   1376.93            2051               443
      sasuke423       5433345.0                    707.53               0                 1
          Fovez       3547267.0                  30197.12             802                 3
      Confucius       3287576.0                 165535.90             719              

In [18]:
# NEW D: usernames that start with "deactivated"
deactivated = users[users["username"].str.lower().str.startswith("deactivated")]
print("usernames starting with 'deactivated':", len(deactivated))
if "has_rows" in users:
    print("  of them with at least one list row:", int(deactivated["has_rows"].sum()))
print(deactivated[["username", "user_completed", "user_plantowatch", "user_days_spent_watching"]].head(10).to_string(index=False))


usernames starting with 'deactivated': 10
  of them with at least one list row: 6
             username  user_completed  user_plantowatch  user_days_spent_watching
      deactivatedshio             170                28                     74.16
     deactivated13041               0                 0                      0.00
     deactivated20325               0                 0                      0.00
       Deactivated453              13                26                      8.49
deactivated_yorozuuya              70                 0                     22.98
     deactivated30325               0                 0                      0.00
       deactivated000               7                 6                      2.73
   deactivated-180120             262                88                    108.67
       deactivated106             715               422                    205.91
          Deactivated              56                22                     47.28


In [19]:
# NEW E: declared against actual counts, direction of the differences
if RUN_LISTS:
    for col in count_cols:
        d = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} actual > declared: {int((d > 0).sum()):>6,} | actual < declared: {int((d < 0).sum()):>6,}"
              f" | largest gap: {int(d.abs().max()):,}")

    zero_declared = merged[(declared_total == 0) & (actual_total > 0)]
    print("\nusers that declare 0 anime but have rows:")
    print(zero_declared[[c + "_actual" for c in count_cols]].assign(total_rows=actual_total[zero_declared.index]).to_string())
else:
    print("Skipped (RUN_LISTS = False)")


user_watching      actual > declared:  2,954 | actual < declared:  2,510 | largest gap: 255
user_completed     actual > declared:    329 | actual < declared:  3,482 | largest gap: 1,043
user_onhold        actual > declared:     92 | actual < declared:    171 | largest gap: 215
user_dropped       actual > declared:     93 | actual < declared:    171 | largest gap: 208
user_plantowatch   actual > declared:  1,488 | actual < declared:    391 | largest gap: 1,432

users that declare 0 anime but have rows:
                  user_watching_actual  user_completed_actual  user_onhold_actual  user_dropped_actual  user_plantowatch_actual  total_rows
username                                                                                                                                   
ker0chan                          51.0                  574.0                 0.0                  0.0                   1432.0      2057.0
Sora-Senpai                        0.0                   93.0            

### Nhận xét cho các kiểm tra làm rõ

**A. Giá trị anime.**
- **`rank` và `popularity` bằng 0: mỗi cột 9 anime.** Rank bắt đầu từ 1, nên 0 là placeholder. Đây là con số nhỏ, so với 1,574 anime có `rank` rỗng. Chưa kiểm tra 9 anime này có giống nhau ở cả hai cột hay không.
- **`broadcast`: một từ placeholder.** Toàn bộ 2,241 placeholder là từ `Unknown`, nên một quy tắc là đủ.
- **`duration`: 313 văn bản ta không đọc được là giá trị thật, không phải giá trị thiếu.** Chúng là độ dài viết bằng **giây**: `30 sec.` (54 anime), `30 sec. per ep.` (42), `15 sec.` (30), `15 sec. per ep.` (17), `40 sec. per ep.` (8), `45 sec.` (7), và tiếp tục như vậy. Chín văn bản phổ biến nhất như thế phủ 174 trong 313, và phần còn lại có lẽ cùng loại (chưa in). Code của ta chỉ biết `hr` và `min`. Các clip rất ngắn và quảng cáo có độ dài như vậy. Chỉ 405 giá trị `Unknown` (2.8%) là thiếu thật, không phải 718 (5.0%). Quy tắc cho bước 4: đọc cả `sec`, và chia cho 60.
- **`score = 0` dù có người chấm: 46 anime, và mười anime có nhiều người chấm nhất đều là `Not yet aired`.** Chúng có 4 đến 15 người chấm và `members` lớn (ví dụ *Tokyo Ghoul:re 2nd Season*, *Overlord III*, *Shingeki no Kyojin Season 3* và *One Punch Man 2*, mỗi anime có 28,000 đến 234,000 members). Ở phần 1 của notebook 02 ta đếm 46 anime có nhãn `Not yet aired` mà có người chấm, nên rất có thể đây là cùng 46 anime đó (chưa kiểm tra). Cách hiểu: **điểm không được công bố trước khi anime phát sóng**, nên `score = 0` ở đây nghĩa là "chưa có điểm công bố", không phải "tệ". Điều này khớp với kế hoạch bỏ các anime `Not yet aired` khỏi các câu hỏi về điểm.

**B. Ngày và tuổi của user.**
- **Tất cả 89,283 giá trị `last_online` trước năm 2000 đều đúng vào năm 1900.** Ngày thật sẽ trải trên nhiều năm. Các user có năm thật đi từ 638 vào năm 2007 lên 40,663 vào năm 2017 và 39,354 vào năm 2018 (đến tháng 5). Vậy năm 1900 là một giá trị mặc định, và năm có lẽ đã bị mất, ủng hộ giả thuyết của ta. Tháng, ngày và giờ của các giá trị này có thể vẫn đúng, nhưng **năm thật thì không biết**. Các ví dụ ta thấy là từ tháng 3 đến tháng 5, và thay 1900 bằng 2018 sẽ chỉ là đoán. Với 29.5% số user này, `last_online` không dùng được làm ngày.
- **8 văn bản `birth_date` mà pandas không đọc được là ngày giả:** các năm 1, 1337, 6969 và 9001 (pandas chỉ giữ các năm từ 1677 đến 2262).
- **User dưới 8 tuổi (769): năm sinh trải từ 2010 đến 2018**, (tám năm phổ biến nhất có 71 đến 91 user mỗi năm, và 82 user sinh năm 2018, nên tuổi của họ là 0). Không có một giá trị mặc định duy nhất. Đó là ngày sai hoặc ngày đùa.
- **User trên 80 tuổi (955): 842 người (88.2%) sinh năm 1930.** Một năm chứa gần như tất cả trông như một **giá trị mặc định**, ví dụ năm đầu tiên của một danh sách thả xuống khi user đăng ký. Đây là giả thuyết. Chỉ 113 user trên 80 tuổi có năm sinh khác.

**C. Giá trị cực đoan, và đặt mức chặn ở đâu.**

| Cột | Số user vượt qua giá trị | |
|---|---|---|
| `stats_episodes` | 10,000: 19,176 / 50,000: 602 / 100,000: 204 / 1,000,000: 20 | |
| `user_days_spent_watching` | 365: 1,673 / 1,000: 192 / 3,650: 42 / 36,500: 3 | |

- **Thang đo.** Với 24 phút mỗi tập, 1,000,000 tập là khoảng 46 năm xem liên tục, và 100,000 tập là khoảng 4.6 năm. Vậy 20 user trên 1,000,000 là không thể, và 204 user trên 100,000 rất đáng ngờ. 3,650 ngày là 10 năm xem liên tục, và 42 user vượt qua mức đó.
- **Các giá trị lớn nhất đến từ số tập vô lý, không phải việc xem thật.** `warrose` có 243,794,881 tập và 2,601,351 ngày, với 2,034 anime completed. `mmmapples` có 18,218,986 tập và 23,640 ngày mà **không có anime completed hay planned nào**. `sasuke423` có 5,433,345 tập với 0 anime completed. Hai trong các giá trị lớn nhất nằm gần đúng 2^24 = 16,777,216: `darkraigiratina` có 16,776,960 và `Kaiyuso` 16,791,524. Đây là cùng loại giới hạn lưu trữ mà ta thấy ở `my_rewatching_ep` trong bước 2, nên các giá trị này có lẽ là một bộ đếm bị tràn.
- **Hai số tóm tắt không phải lúc nào cũng khớp nhau.** `warrose` có 15 phút xem mỗi tập, là bình thường. `Kitayama_Naoki` có 10,989,789 tập nhưng chỉ 1,377 ngày, tức 11 giây mỗi tập. Vậy ta không thể sửa các user này bằng cách tính lại một cột từ cột kia.
- **Đề xuất cho bước 4 (nhóm quyết định).** Chặn `stats_episodes` và `user_days_spent_watching` ở phân vị 99.9 của chúng (77,358 tập và 758 ngày, từ phần 3), hoặc loại các user trên 100,000 tập (204 user, tức 0.07%), và báo cáo con số này.

**D. Username bắt đầu bằng `deactivated`: chỉ riêng cái tên không giải thích gì.** Có 10 username như vậy, và **6 trong số đó vẫn sở hữu dòng trong list** (ví dụ `deactivatedshio` với 170 anime completed). Trong 4 user không có dòng, ba người khai 0 anime (tài khoản rỗng) và một người là `deactivated-180120`, khai 262 anime completed. Vậy giả thuyết "tài khoản bị vô hiệu hóa và list bị xóa" không được ủng hộ: hầu hết tài khoản deactivated vẫn giữ các dòng. **Ta không giải thích được bất kỳ ai trong 8 user khai có anime mà không có dòng.**

**E. Số khai báo so với số thật: chiều lệch không ngẫu nhiên.**

| Trạng thái | Số thật cao hơn số khai báo | Số thật thấp hơn số khai báo |
|---|---|---|
| watching | 2,954 | 2,510 |
| completed | 329 | 3,482 |
| on hold | 92 | 171 |
| dropped | 93 | 171 |
| plan to watch | 1,488 | 391 |

- **Với `completed`, list có ít dòng hơn profile nói trong 91% số trường hợp không khớp (3,482 trong 3,811), và với `plan to watch` list có nhiều dòng hơn trong 79% (1,488 trong 1,879).** Hãy nghĩ đến những user xem xong các anime họ định xem: theo thời gian `completed` tăng lên và `plan to watch` giảm đi. Mẫu này là điều ta thấy nếu **list được đọc sớm hơn profile**: profile (mới hơn) cho thấy nhiều anime completed hơn và ít anime planned hơn các dòng (cũ hơn). `watching` cân bằng (2,954 so với 2,510). Đây là một giả thuyết về thứ tự hai lần đọc, nhưng nó giải thích cả ba chiều bằng một câu chuyện. Nó cũng giải thích vì sao các chênh lệch nhỏ với hầu hết user.
- **Một phần các chênh lệch lớn nhất là các user không có dòng.** Chênh lệch lớn nhất của `completed` (1,043) là `Useless_Goddess`, khai 1,043 anime completed và không có dòng nào.
- **6 user khai 0 anime nhưng có dòng có list dài.** `ker0chan` có 2,057 dòng, `Omiyage` 895, `AliNyman` 565, `Sora-Senpai` 335, `Ahngel` 294 và `4895624815796354` 294, tổng cộng 4,440 dòng. Số đếm profile của họ là 0 dù list không rỗng, và câu chuyện "list cũ hơn" ở trên không giải thích được điều này. Với 6 user này, số khai báo không dùng được.


## 7. Những kiểm tra mà phần 1 đến 6 chưa làm: bảng list, các cột số, username, trùng lặp và mâu thuẫn (MỚI)

**Mục đích.** Khi đọc lại phần 1 đến 6 như một tổng thể, ta tìm thấy những chỗ ta chỉ nhìn một bảng hay một loại cột mà không nhìn các loại khác, hoặc mô tả một cột mà không kiểm tra nó. Phần này chạy các kiểm tra đó, để cuộc kiểm tra không còn điểm mù trước các quyết định làm sạch của bước 4. Nó không thay đổi gì trong dữ liệu và dùng lại `anime`, `users`, `paths` và `RUN_LISTS` từ các cell ở trên.

**Cách chạy.** Phần 7A chỉ đọc hai bảng nhỏ (vài giây). Phần 7B đọc bảng list 80M dòng một lần, mỗi lần 3M dòng (khoảng 4 phút); `RUN_LISTS = False` bỏ qua nó. Lượt đọc và các báo cáo nằm ở các cell riêng, nên một lỗi trong báo cáo không cần đọc lại lượt thứ hai.

**Những gì còn thiếu, và vì sao quan trọng.**

| # | Chỗ hổng trong notebook 03 | Vì sao quan trọng | Ở đâu |
|---|---|---|---|
| 1 | Bảng list (80M dòng, bảng lớn nhất) chưa từng được quét tìm placeholder. Lần nhìn qua đầu tiên (1a) cố ý bỏ qua vì kích thước. | Placeholder trong bảng lớn nhất ảnh hưởng nhiều dòng nhất. `my_tags` là văn bản tự do, và hai cột ngày có thể chứa ngày mặc định hoặc ngày không thể có | B1 đến B3 |
| 2 | Các cột số chưa từng được quét tìm số placeholder (`-1`, `999`, `9999`). Lần nhìn qua đầu tiên (1a) chỉ đọc các cột văn bản. | Một số như `-1` hay `9999` đại diện cho "không biết" sẽ lọt vào mọi phép trung bình và tương quan | A1 |
| 3 | Username được kiểm tra là duy nhất (khớp chính xác), nhưng không kiểm tra **chữ hoa và khoảng trắng**. | Username là khóa join giữa `UserList` và `UserAnimeList`. Hai cách viết của một tên, hoặc một khoảng trắng ở cuối, làm hỏng phép join mà không báo lỗi | A2, B4 |
| 4 | `location` chỉ được đếm theo các giá trị phổ biến nhất. | Nếu `USA`, `usa` và `Usa ` là ba giá trị, mọi phép gom nhóm theo quốc gia bị chia nhỏ | A3 |
| 5 | Trùng lặp chỉ được kiểm tra với `anime_id`. | Cùng một anime có thể tồn tại dưới hai id, làm trọng số của nó tăng gấp đôi | A4 |
| 6 | Không kiểm tra hai cột của cùng một dòng có khớp nhau không (`scored_by` so với `members`, ngày kết thúc so với ngày bắt đầu, anime đã kết thúc mà bắt đầu sau thời điểm crawl). | Một dòng mâu thuẫn với chính nó là sai ở ít nhất một cột | A5, A5b, A5c, B3 |
| 7 | Các cột chứa văn bản Python (`aired`, `related`, `opening_theme`, `ending_theme`) được đọc bằng `ast.literal_eval` trong notebook 02 mà không đếm số lỗi. | Một lỗi đọc âm thầm loại một dòng khỏi mọi phân tích của các cột này | A6 |
| 8 | Các ngày của bảng list được xem như một danh sách top 5 trong notebook 02, và chưa từng được kiểm tra ngày một phần, ngày không thể có, ngày sau thời điểm crawl, hay ngày kết thúc trước ngày bắt đầu. `my_last_updated` chỉ được dùng để lấy giá trị lớn nhất. | Khoảng ngày quyết định những dòng nào dùng được cho một phân tích theo thời gian | B3 |
| 9 | `anime_id` của bảng list chưa từng được kiểm tra với `AnimeList`, và `my_watched_episodes` chưa từng được so với số tập. | Một id không có anime thì không join được. Xem nhiều tập hơn số tập tồn tại là không thể | B4, B5 |

**Nhãn bằng chứng.** Mọi thứ được in ra là số đếm từ các file ([V]). Giải thích *vì sao* một giá trị tồn tại là giả thuyết và được ghi rõ như vậy.


In [20]:
# Setup for part 7 (the objects `anime`, `users`, `paths`, `na_kwargs` and `RUN_LISTS` come from the cells above)
from collections import Counter
import ast, html, re

CRAWL_END = pd.Timestamp("2018-05-25", tz="UTC")  # estimated end of the crawl (notebook 02)


### 7A. Hai bảng nhỏ

### A1. Các cột số: giá trị không thể có và số placeholder

Với mỗi cột số (bỏ qua các cột id), cell in ra số ô thiếu, giá trị nhỏ nhất, giá trị lớn nhất, số giá trị âm, năm giá trị phổ biến nhất, và số ô bằng một số thường dùng làm placeholder (`-1`, `99`, `999`, `9999`, `99999`). Một giá trị như `999` cũng có thể là thật (ví dụ số lượt yêu thích), nên một kết quả trùng chỉ là lý do để xem xét, không phải bằng chứng.


In [21]:
# A1. Numeric columns: impossible values, most frequent values, placeholder-like numbers
SENTINELS = [-1, 99, 999, 9999, 99999]

def numeric_overview(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(include="number").columns:
        if col.endswith("_id"):
            continue
        s = df[col].dropna()
        top = " | ".join(f"{v:,.6g} x{c:,}" for v, c in s.value_counts().head(n).items())
        sentinel = {v: int((s == v).sum()) for v in SENTINELS if (s == v).any()}
        extra = f" | placeholder-like: {sentinel}" if sentinel else ""
        print(f"{col}: missing {int(df[col].isna().sum()):,} | min {s.min():,.6g} | max {s.max():,.6g}"
              f" | negative {int((s < 0).sum()):,} | top: {top}{extra}")

numeric_overview(anime, "anime table")
print()
numeric_overview(users, "user table")


===== anime table
episodes: missing 0 | min 0 | max 1,818 | negative 0 | top: 1 x6,857 | 2 x1,187 | 12 x1,034 | 13 x637 | 3 x558 | placeholder-like: {99: 2}
score: missing 0 | min 0 | max 10 | negative 0 | top: 0 x354 | 6 x120 | 5 x100 | 6.5 x90 | 6.33 x87
scored_by: missing 0 | min 0 | max 1.00948e+06 | negative 0 | top: 0 x308 | 8 x155 | 7 x143 | 5 x140 | 9 x139 | placeholder-like: {99: 11, 999: 2}
rank: missing 1,574 | min 0 | max 12,919 | negative 0 | top: 0 x9 | 6,303 x5 | 3,012 x4 | 12,630 x4 | 4,700 x4 | placeholder-like: {99: 1, 999: 1, 9999: 1}
popularity: missing 0 | min 0 | max 14,487 | negative 0 | top: 0 x9 | 3,252 x4 | 6,104 x4 | 2,870 x4 | 9,537 x4 | placeholder-like: {99: 1, 999: 1}
members: missing 0 | min 0 | max 1.45638e+06 | negative 0 | top: 123 x35 | 87 x35 | 85 x32 | 115 x32 | 81 x30 | placeholder-like: {99: 17}
favorites: missing 0 | min 0 | max 106,895 | negative 0 | top: 0 x5,018 | 1 x1,539 | 2 x817 | 3 x577 | 4 x407 | placeholder-like: {99: 6, 999: 1}

===== 

### A2. Khoảng trắng, chữ hoa và username (khóa join)

Cell đầu đếm, trong mọi cột văn bản của cả hai bảng, các ô có khoảng trắng ở đầu hoặc ở cuối, và các ô có hai khoảng trắng trở lên liền nhau. Cell thứ hai kiểm tra username: có những tên chỉ khác nhau ở chữ hoa hoặc một khoảng trắng không? Trên MyAnimeList địa chỉ profile không phụ thuộc chữ hoa, nên hai tên như vậy có thể là cùng một người; và một phép join trên văn bản chính xác sẽ bỏ sót họ.


In [22]:
# A2. Spaces in text cells, and usernames that differ only by capital letters
def spacing(df, label):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        edge = int((s != s.str.strip()).sum())
        double = int(s.str.contains(r"\s{2,}", regex=True).sum())
        if edge or double:
            print(f"{col}: space at the start or the end {edge:,} | two or more spaces in a row {double:,}")

spacing(anime, "anime table")
print()
spacing(users, "user table")

name = users["username"]
low = name.str.lower()
clash = low.duplicated(keep=False)
print()
print("usernames:", f"{len(name):,}", "| distinct:", f"{name.nunique():,}", "| distinct after lowercase:", f"{low.nunique():,}")
print("names that differ only by capital letters:", int(clash.sum()), "in", low[clash].nunique(), "groups")
print(users.loc[clash, ["username", "user_id"]].sort_values("username", key=lambda s: s.str.lower()).head(10).to_string(index=False))
print("usernames with a space at the start or the end:", int((name != name.str.strip()).sum()))


===== anime table
title: space at the start or the end 0 | two or more spaces in a row 3
title_japanese: space at the start or the end 4 | two or more spaces in a row 32
title_synonyms: space at the start or the end 0 | two or more spaces in a row 5
background: space at the start or the end 105 | two or more spaces in a row 29
related: space at the start or the end 0 | two or more spaces in a row 2
opening_theme: space at the start or the end 0 | two or more spaces in a row 40
ending_theme: space at the start or the end 0 | two or more spaces in a row 82

===== user table
location: space at the start or the end 4,560 | two or more spaces in a row 406

usernames: 302,675 | distinct: 302,675 | distinct after lowercase: 302,675
names that differ only by capital letters: 0 in 0 groups
Empty DataFrame
Columns: [username, user_id]
Index: []
usernames with a space at the start or the end: 0


### A3. `location`: có bao nhiêu cách viết ẩn sau cùng một địa điểm?

`location` là văn bản tự do. Cell so sánh số văn bản thô khác nhau với số lượng sau khi ta bỏ chữ hoa và khoảng trắng thừa, và in ra các địa điểm có nhiều cách viết nhất. Việc này cho biết bước 4 có bao nhiêu việc gom nhóm trước mọi phân tích theo quốc gia hay thành phố.


In [23]:
# A3. location: raw texts against texts without capital letters and extra spaces
loc = users["location"].dropna()
norm = loc.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
print("filled locations:", f"{len(loc):,}", "| distinct raw texts:", f"{loc.nunique():,}", "| distinct after normalising:", f"{norm.nunique():,}")
print("most frequent after normalising:", norm.value_counts().head(10).to_dict())

spellings = loc.groupby(norm).nunique().sort_values(ascending=False)
print("\nplaces written in the most different ways:")
for key in spellings.head(5).index:
    forms = loc[norm == key].value_counts().head(6).to_dict()
    print(f"  {key!r}: {int(spellings[key])} spellings, for example {forms}")


filled locations: 156,799 | distinct raw texts: 55,283 | distinct after normalising: 49,616
most frequent after normalising: {'poland': 2544, 'germany': 2004, 'canada': 1946, 'california': 1679, 'brazil': 1517, 'philippines': 1281, 'france': 1204, 'australia': 1101, 'sweden': 1080, 'singapore': 952}

places written in the most different ways:
  'saudi arabia': 12 spellings, for example {'Saudi Arabia': 82, 'saudi arabia': 19, 'Saudi Arabia ': 12, ' Saudi Arabia': 5, 'saudi arabia ': 3, 'Saudi arabia': 2}
  'new york': 12 spellings, for example {'New York': 661, 'new york': 45, 'New York ': 24, 'New york': 15, ' New York': 4, 'NEW YORK': 3}
  'los angeles, california': 11 spellings, for example {'Los Angeles, California': 325, 'Los Angeles, California ': 16, 'los angeles, california': 7, 'Los angeles, California': 3, 'los angeles, california ': 1, 'Los Angeles, California     ': 1}
  'somewhere over the rainbow': 11 spellings, for example {'Somewhere over the rainbow': 31, 'somewhere ov

### A4. Trùng lặp ngoài `anime_id`

Notebook 03 chỉ kiểm tra `anime_id` là duy nhất. Ở đây ta tìm cùng một tựa dưới nhiều id, và các dòng giống hệt nhau ở mọi cột trừ `anime_id` (cùng một anime nhập hai lần).


In [24]:
# A4. Duplicate anime that have different ids
dup_title = anime[anime["title"].duplicated(keep=False)]
print("titles used by more than one anime_id:", dup_title["title"].nunique(), "| rows:", len(dup_title))
print(dup_title.sort_values("title")[["anime_id", "title", "type", "status", "scored_by"]].head(10).to_string(index=False))

same_but_id = anime.drop(columns=["anime_id"]).duplicated(keep=False)
print("\nrows identical in every column except anime_id:", int(same_but_id.sum()))

title_clean = anime["title"].str.strip().str.lower()
print("titles that differ only by capital letters or spaces:", int(title_clean.duplicated(keep=False).sum() - anime["title"].duplicated(keep=False).sum()))


titles used by more than one anime_id: 1 | rows: 2
 anime_id       title type           status  scored_by
    35102 Hinamatsuri  OVA  Finished Airing          8
    36296 Hinamatsuri   TV Currently Airing      12470

rows identical in every column except anime_id: 0
titles that differ only by capital letters or spaces: 0


### A5. Các cột trong cùng một dòng có khớp nhau không?

Một dòng mâu thuẫn với chính nó là sai ở ít nhất một cột. Cell đếm: `score` ngoài khoảng 0 đến 10; số người chấm nhiều hơn số members (`scored_by` > `members`); số lượt yêu thích nhiều hơn số members; ngày kết thúc trước ngày bắt đầu trong `aired`; anime **đã kết thúc** mà ngày bắt đầu nằm sau thời điểm crawl; và anime **đang phát sóng** mà ngày kết thúc nằm trước thời điểm crawl (crawl kết thúc khoảng 2018-05-25, xem notebook 02).


In [25]:
# A5. Contradictions inside one row of the anime table
def aired_part(t, key):
    try:
        return pd.to_datetime(ast.literal_eval(t).get(key), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

air_start = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "from")), utc=True)
air_end = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "to")), utc=True)

print("score outside 0 to 10:", int(((anime["score"] < 0) | (anime["score"] > 10)).sum()))
print("scored_by > members:", int((anime["scored_by"] > anime["members"]).sum()),
      "| favorites > members:", int((anime["favorites"] > anime["members"]).sum()))
print("end date before start date:", int((air_end < air_start).sum()))
finished_late = (anime["status"] == "Finished Airing") & (air_start > CRAWL_END)
airing_over = (anime["status"] == "Currently Airing") & (air_end < CRAWL_END)
print("Finished Airing with a start date after the crawl:", int(finished_late.sum()))
print("Currently Airing with an end date before the crawl:", int(airing_over.sum()))
cols = ["anime_id", "title", "status", "aired_string"]
print(anime.loc[finished_late, cols].head(5).to_string(index=False))
print(anime.loc[airing_over, cols].head(5).to_string(index=False))


score outside 0 to 10: 0
scored_by > members: 0 | favorites > members: 1
end date before start date: 32
Finished Airing with a start date after the crawl: 16
Currently Airing with an end date before the crawl: 6
 anime_id                    title          status              aired_string
    21277            Yawaraka Atom Finished Airing    Oct, 2007 to Oct, 2008
    33756               Hen Xi You Finished Airing Jun, 2016 to Nov 28, 2016
    13819         Kamiusagi Rope 2 Finished Airing    Nov, 2010 to Dec, 2011
    19695               Clover 4/3 Finished Airing    Aug, 2000 to Dec, 2000
    28733 Ginga Tetsudou 999 (ONA) Finished Airing    Sep, 2002 to Jul, 2004
 anime_id             title           status                 aired_string
    36623        SNS Police Currently Airing Mar 24, 2018 to May 19, 2018
    35857 Lupin III: Part 5 Currently Airing          Apr 4, 2018 to 2018
    36902  Mahou Shoujo Ore Currently Airing          Apr 2, 2018 to 2018
    34572      Black Clover Cu

### A5b. Văn bản `aired` thô của các dòng tự mâu thuẫn (MỚI)

A5 tìm thấy các dòng tự mâu thuẫn, nhưng chỉ in `aired_string`. Với năm anime "Finished Airing" đầu tiên, `aired_string` cho thấy một ngày cũ (ví dụ `Oct, 2007 to Oct, 2008`), vậy mà A5 nói ngày bắt đầu của chúng trong `aired` nằm sau thời điểm crawl. Vậy `aired` và `aired_string` dường như không khớp nhau với chúng. Cell này in văn bản `aired` thô của từng nhóm, và kiểm tra một phỏng đoán: **khi phần kết thúc của `aired_string` chỉ là một năm hoặc một tháng, `aired` lưu ngày đầu tiên của khoảng đó**, điều này sẽ làm ngày kết thúc nằm trước ngày bắt đầu (ví dụ `Apr 4, 2018 to 2018` sẽ kết thúc vào 2018-01-01).


In [26]:
# A5b. The raw `aired` text of the anime that contradict themselves, and the guess about partial end dates
def end_kind(t):
    parts = str(t).split(" to ")
    if len(parts) < 2: return "no end text"
    e = parts[1].strip()
    if e == "?": return "still open"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", e): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", e): return "month and year"
    if re.search(r"^\d{4}$", e): return "year only"
    return "other"

groups = {
    "end date before start date": air_end < air_start,
    "Finished Airing, start date after the crawl": finished_late,
    "Currently Airing, end date before the crawl": airing_over,
    "favorites > members": anime["favorites"] > anime["members"],
}
for label, mask in groups.items():
    print(f"--- {label}: {int(mask.sum())} anime")
    print(anime.loc[mask, ["anime_id", "title", "status", "aired_string", "aired"]].head(40).to_string(index=False, max_colwidth=75))

has_end = air_end.notna()
kinds = anime["aired_string"].map(end_kind)
before_start = groups["end date before start date"]
print()
print("how the end of `aired_string` is written, for the anime that have an end date in `aired`:")
print("  all:", kinds[has_end].value_counts().to_dict())
print("  end before start:", kinds[before_start].value_counts().to_dict())
print(f"end date on the 1st of a month: all {(air_end[has_end].dt.day == 1).mean():.1%} | end before start {(air_end[before_start].dt.day == 1).mean():.1%}")


--- end date before start date: 32 anime
 anime_id                                                   title          status                 aired_string                                      aired
    33600                                          Menghuan Xiyou Finished Airing  Dec 29, 2014 to Feb 6, 2015 {'from': '2015-12-29', 'to': '2015-02-06'}
     6822                                  Ukkari Pénélope (2009) Finished Airing  Jun 29, 2009 to Jan 2, 2010 {'from': '2009-06-29', 'to': '2009-01-02'}
    33126                   Kurogane no Linebarrels Picture Drama Finished Airing Dec 14, 2009 to Aug 26, 2009 {'from': '2009-12-14', 'to': '2009-08-26'}
    21195                             Soul Eater: Late Night Show Finished Airing       Apr, 2008 to Mar, 2009 {'from': '2018-04-10', 'to': '2018-03-10'}
    30840                                   Many Things SD Gundam Finished Airing         2011 to Jul 22, 2011 {'from': '2018-05-11', 'to': '2011-07-22'}
    34234                          

### A5c. Có bao nhiêu ngày `aired` được điền vào? (MỚI)

A5b cho thấy một ngày chỉ được viết một phần trong `aired_string` (`Oct, 2007`, `2011`) được lưu trong `aired` thành một ngày đầy đủ, và các con số giống nhau cứ quay lại: năm 2018, tháng 5, và các ngày 9 đến 12. Giả thuyết của ta là các phần còn thiếu được điền bằng ngày lấy dữ liệu (fetch). Cell này kiểm tra giả thuyết đó trên **mọi** anime, không chỉ 54 dòng mà A5b đã in.

Với phần bắt đầu và phần kết thúc của `aired_string`, nó cho biết văn bản được viết thế nào (ngày đầy đủ, tháng và năm, chỉ năm, `?` cho kết thúc còn mở, hoặc vắng mặt). Với mỗi loại, nó đếm số anime có ngày trong `aired` và in ra các ngày, tháng và năm phổ biến nhất của những ngày đó. Với các ngày đầy đủ, nó cũng đếm bao lần ngày trong `aired` khác với ngày viết trong văn bản (ví dụ lệch một năm).


In [27]:
# A5c. Are the missing parts of a date in aired_string filled in `aired`? (all anime)
def split_aired(t):
    p = str(t).split(" to ")
    return p[0].strip(), (p[1].strip() if len(p) > 1 else None)

def text_kind(e):
    if pd.isna(e): return "absent"
    if e == "?": return "open"
    if re.search(r"^[A-Z][a-z]{2} \d{1,2}, \d{4}$", e): return "full date"
    if re.search(r"^[A-Z][a-z]{2}, \d{4}$", e): return "month and year"
    if re.search(r"^\d{4}$", e): return "year only"
    return "other"

pairs = anime["aired_string"].map(split_aired)
for label, pos, stamp in (("start", 0, air_start), ("end", 1, air_end)):
    texts = pairs.map(lambda p: p[pos])
    text_kinds = texts.map(text_kind)
    print(f"--- {label}: how `aired_string` writes it, and what `aired` holds")
    for k in ["full date", "month and year", "year only", "open", "absent", "other"]:
        in_kind = text_kinds == k
        has_date = in_kind & stamp.notna()
        line = f"  {k:15s} anime: {int(in_kind.sum()):>6,} | with a date in aired: {int(has_date.sum()):>6,}"
        if has_date.any():
            s = stamp[has_date]
            line += (f" | day top 4: {s.dt.day.value_counts().head(4).to_dict()}"
                     f" | month top 3: {s.dt.month.value_counts().head(3).to_dict()}"
                     f" | year top 3: {s.dt.year.value_counts().head(3).to_dict()}")
        print(line)
    is_full = text_kinds == "full date"
    written = pd.to_datetime(texts.where(is_full), format="%b %d, %Y", errors="coerce")
    same = stamp.dt.tz_localize(None).dt.normalize() == written
    compared = is_full & stamp.notna() & written.notna()
    print(f"  full dates in the text that also have a date in aired: {int(compared.sum()):,} | aired differs from the text: {int((compared & ~same).sum()):,}")
    print(anime.loc[compared & ~same, ["anime_id", "aired_string", "aired"]].head(8).to_string(index=False, max_colwidth=70))
    print()


--- start: how `aired_string` writes it, and what `aired` holds
  full date       anime: 12,560 | with a date in aired: 12,560 | day top 4: {25: 799, 21: 678, 5: 623, 1: 612} | month top 3: {4: 1959, 10: 1716, 7: 1477} | year top 3: {2016: 794, 2017: 782, 2014: 755}
  month and year  anime:    476 | with a date in aired:    122 | day top 4: {11: 81, 10: 15, 12: 10, 9: 10} | month top 3: {7: 44, 10: 34, 4: 13} | year top 3: {2018: 106, 1992: 3, 1999: 3}
  year only       anime:  1,219 | with a date in aired:    105 | day top 4: {11: 71, 10: 17, 12: 7, 9: 6} | month top 3: {5: 105} | year top 3: {2018: 85, 1992: 3, 1986: 2}
  open            anime:      0 | with a date in aired:      0
  absent          anime:      0 | with a date in aired:      0
  other           anime:    223 | with a date in aired:      0
  full dates in the text that also have a date in aired: 12,560 | aired differs from the text: 52
 anime_id                aired_string                                      aired
  

### A6. Các cột chứa văn bản Python: có bao nhiêu ô không đọc được?

`aired`, `related`, `opening_theme` và `ending_theme` là các văn bản viết giống một dictionary hoặc một list của Python. Notebook 02 đọc chúng bằng `ast.literal_eval`, và một ô bị lỗi thì âm thầm biến thành giá trị thiếu. Ở đây các lỗi được đếm.


In [28]:
# A6. Python-literal columns: how many cells fail to parse?
for col in ["aired", "related", "opening_theme", "ending_theme"]:
    s = anime[col].dropna()
    bad = []
    for v in s:
        try:
            ast.literal_eval(v)
        except Exception:
            bad.append(v)
    example = bad[0][:80] if bad else "-"
    print(f"{col:14s} cells: {len(s):>6,} | cannot be parsed: {len(bad):>5,} | example: {example!r}")


aired          cells: 14,478 | cannot be parsed:     0 | example: '-'
related        cells: 14,478 | cannot be parsed:     0 | example: '-'
opening_theme  cells: 14,478 | cannot be parsed:     0 | example: '-'
ending_theme   cells: 14,478 | cannot be parsed:     0 | example: '-'


### Nhận xét cho phần 7A

**Cách đọc.** Mỗi khối dưới đây trả lời một kiểm tra. "Đã loại trừ" nghĩa là ta đã tìm vấn đề đó và không thấy.

**A1. Các cột số: không có số placeholder (đã loại trừ).**
- Không cột số nào có giá trị âm.
- Các kết quả "giống placeholder" (`99`, `999`, `9999`) là báo động giả. Chúng là các số đếm bình thường: 17 anime có đúng 99 members và 891 user có đúng 99 anime completed. Mọi số nguyên đều xuất hiện trong một cột đếm, nên `99` tự hiện ra.
- Các giá trị phổ biến nhất chỉ cho thấy những số 0 mà ta đã biết: `score = 0` với 354 anime, `rank = 0` và `popularity = 0` mỗi cột 9 anime, `favorites = 0` với 5,018 anime (34.7%), `stats_mean_score = 0` với 30,922 user.
- `access_rank` thiếu với cả 302,675 user. `stats_mean_score`, `stats_rewatched` và `stats_episodes` thiếu với 129 user, cùng con số với các profile rỗng của notebook 02 (ta chưa kiểm tra họ có phải cùng những user đó không).
- Các giá trị lớn nhất đã được thấy trong notebook 03: `user_days_spent_watching` lên đến 2,601,350 ngày (khoảng 7,000 năm) và `stats_episodes` lên đến 243,795,000 là không thể. `episodes` lên đến 1,818 là hợp lý với một series rất dài.

**A2. Khoảng trắng và username.**
- **`location` có khoảng trắng ở đầu hoặc cuối trong 4,560 ô (2.9% trong 156,799 ô có giá trị)** và hai khoảng trắng liền nhau trong 406 ô. Các cột khác chỉ có ít: `background` 105 và `title_japanese` 4 (ở rìa), `opening_theme` 40, `ending_theme` 82, `title_japanese` 32 và `background` 29 (hai khoảng trắng). Những cái này nhỏ.
- **Username sạch (đã loại trừ).** 302,675 tên khác nhau, vẫn là 302,675 sau khi đổi sang chữ thường, không tên nào có khoảng trắng ở đầu hoặc cuối. Không có hai tên nào chỉ khác nhau ở chữ hoa, nên khóa join không có bản trùng ẩn.

**A3. `location`: một phần mười số cách viết biến mất khi chuẩn hóa.**
- 55,283 văn bản thô thành 49,616 sau khi bỏ chữ hoa và khoảng trắng thừa: ít hơn 5,667 (10.3%).
- Phổ biến nhất sau khi chuẩn hóa: `poland` 2,544, `germany` 2,004, `canada` 1,946, `california` 1,679, `brazil` 1,517, `philippines` 1,281. Một tiểu bang (`california`) đứng giữa các quốc gia, nên gom nhóm theo quốc gia cần một bảng ánh xạ mà ta phải quyết định ở bước 4. `los angeles, california` là một thành phố nằm trong một tiểu bang.
- Các địa điểm có nhiều cách viết nhất là `saudi arabia` (12), `new york` (12) và `los angeles, california` (11). Sự khác biệt là chữ hoa và khoảng trắng ở rìa (`New York `, ` New York`, `NEW YORK`). Một số location là trò đùa (`Somewhere over the rainbow`, 11 cách viết).
- **Kết luận:** cắt khoảng trắng, đổi chữ thường và gộp khoảng trắng là một bước rẻ loại bỏ 10% giá trị khác nhau. Việc ánh xạ từ địa điểm sang quốc gia là một quyết định riêng.

**A4. Anime trùng lặp (đã loại trừ).** Một tựa xuất hiện hai lần: `Hinamatsuri` là `anime_id` 35102 (một OVA, đã kết thúc) và 36296 (một TV series, đang phát sóng, 12,470 người chấm). Chúng là hai anime khác nhau. Không dòng nào giống hệt dòng khác trừ `anime_id`, và không có tựa nào chỉ khác nhau ở chữ hoa hay khoảng trắng. Tựa không phải là khóa, nên không bao giờ join anime theo tựa.

**A5. Mâu thuẫn trong cùng một dòng: phần lớn là các ngày được điền vào.**
- Đã loại trừ: `score` ngoài khoảng 0 đến 10 (0 anime) và `scored_by` > `members` (0 anime).
- `favorites` > `members`: 1 anime, `Force of Will the Movie` (`anime_id` 33438), chưa phát sóng và không có ngày. Vô hại.
- **Ngày kết thúc trước ngày bắt đầu: 32 anime. "Finished Airing" mà ngày bắt đầu sau thời điểm crawl: 16. "Currently Airing" mà ngày kết thúc trước thời điểm crawl: 6.** Cell A5b đã in văn bản `aired` thô của tất cả chúng (54 dòng), và nó thay đổi cách đọc ba số đếm này.

**A5b cho thấy gì.** Điều này được đọc trên các dòng đã in. A5c, bên dưới, đếm nó trên toàn bảng.
1. **Một ngày chỉ được viết một phần trong `aired_string` được lưu trong `aired` thành một ngày đầy đủ, với những giá trị không có trong văn bản.** Các con số giống nhau cứ quay lại: năm **2018**, tháng **5**, và các ngày **9, 10, 11 và 12**. Ví dụ: `Oct, 2007 to Oct, 2008` được lưu thành `from 2018-10-11, to 2018-10-11`; `Dec, 2005 to Nov, 2011` thành `from 2018-12-11, to 2018-11-11`; `Apr, 2008 to Mar, 2009` thành `from 2018-04-10, to 2018-03-10`; `2011 to Jul 22, 2011` thành `from 2018-05-11`; `Jul 5, 1992 to 1992` thành `to 1992-05-09`; `Apr 4, 2018 to 2018` thành `to 2018-05-10`.
2. **Giả thuyết:** năm, tháng và ngày còn thiếu được điền bằng ngày lấy dữ liệu (fetch). Khi đó bảng anime sẽ được lấy quanh **9 đến 12 tháng 5 năm 2018**, sớm hơn các bảng user (22 đến 25 tháng 5, notebook 02). Đây là một suy luận từ các dòng này. Quy tắc điền chính xác chưa được tìm ra: với một năm bị thiếu, năm được điền đôi khi là 2018 và đôi khi là năm của văn bản.
3. **16 anime "Finished Airing" có ngày bắt đầu sau thời điểm crawl không phải là mâu thuẫn của nguồn.** Cả 16 đều có ngày bắt đầu viết thành tháng và năm (`Oct, 2007`, `Jun, 2016`, ...), và `aired` chứa một ngày bắt đầu trong năm 2018 với tháng sau tháng 5 (`2018-10-11`, `2018-06-11`, ...). Chúng là các ngày được điền vào.
4. **32 trường hợp "kết thúc trước bắt đầu".** Đọc bằng mắt trên các dòng đã in (không bằng code): 21 là ngày được điền vào (ví dụ `Apr, 2008 to Mar, 2009` được lưu thành `2018-04-10` đến `2018-03-10`). 11 còn lại có ngày đầy đủ trong văn bản. Trong 7 trường hợp, **bản thân văn bản mâu thuẫn** (`Mar 10, 1991 to Mar 10, 1990`, `Oct 2, 2016 to Sep 17, 2016`, `Jul 21, 1994 to Jun 21, 1994`), và trong 4 trường hợp năm trong `aired` lệch một năm so với văn bản (`Dec 29, 2014 to Feb 6, 2015` được lưu thành `from 2015-12-29`, `Jun 29, 2009 to Jan 2, 2010` thành `to 2009-01-02`).
5. **6 anime "Currently Airing" có ngày kết thúc trước thời điểm crawl.** Bốn là các ngày kết thúc được điền vào: `Lupin III: Part 5` (`Apr 4, 2018 to 2018`, lưu thành `to 2018-05-10`), `Mahou Shoujo Ore`, `Black Clover` và `Nyanfuru` (`to Mar, 2018`, lưu thành `2018-03-11`). Hai cái còn lại, `SNS Police` và `IDOLiSH7`, kết thúc vào `May 19, 2018`. Nếu bảng anime được lấy vào 9 đến 12 tháng 5, ngày kết thúc đó nằm sau lần lấy và status của chúng là đúng. So với thời điểm kết thúc crawl của các bảng user (25 tháng 5) thì status đó sẽ trông đã cũ.

**Phỏng đoán của A5b không được ủng hộ.** Một ngày kết thúc chỉ có năm không được lưu thành 1 tháng 1: 4.3% mọi ngày kết thúc rơi vào mùng 1 của một tháng, và 3.1% trong nhóm "kết thúc trước bắt đầu". Các ngày cứ quay lại là 9 đến 12. Văn bản của 32 trường hợp "kết thúc trước bắt đầu" là ngày đầy đủ với 18, tháng và năm với 9 và chỉ năm với 5. Trong 12,287 anime có ngày kết thúc trong `aired`, ngày kết thúc được viết chỉ có năm với 86 và tháng và năm với 60.

**Hệ quả.** `aired` chỉ đáng tin khi văn bản của `aired_string` là một ngày đầy đủ. Một ngày được điền vào có năm, tháng hoặc ngày sai, chủ yếu trong năm 2018, nên không được dùng cho xu hướng theo thời gian hay cho "phát hành trước hay sau thời điểm crawl". Năm phát hành nên lấy từ văn bản, và chỉ khi văn bản có năm. Ở bước 4 những ngày này phải được đánh dấu. A5c cho biết số lượng.

**A5c. Các ngày được điền vào, đếm trên toàn bảng: giả thuyết được ủng hộ.**

| `aired_string` viết thế nào | Số anime | Có ngày trong `aired` | Bốn ngày phổ biến nhất | Tháng của các ngày này | Năm của các ngày này |
|---|---|---|---|---|---|
| **Bắt đầu**, ngày đầy đủ | 12,560 | 12,560 | trải đều trong tháng (25: 799, 21: 678, 5: 623, 1: 612) | trải đều | trải đều (ngày thật) |
| **Bắt đầu**, tháng và năm | 476 | **122** | **11: 81, 10: 15, 12: 10, 9: 10** | 7: 44, 10: 34, 4: 13 | 2018: 106 |
| **Bắt đầu**, chỉ năm | 1,219 | **105** | **11: 71, 10: 17, 12: 7, 9: 6** | **Tháng 5 với cả 105** | 2018: 85 |
| **Kết thúc**, ngày đầy đủ | 6,467 | 6,467 | trải đều | trải đều | trải đều |
| **Kết thúc**, tháng và năm | 60 | **60** | **11: 44, 9: 9, 10: 5, 12: 1** | 3: 11, 12: 10, 11: 7 | 2018: 50 |
| **Kết thúc**, chỉ năm | 86 | **86** | **11: 65, 9: 8, 10: 8, 12: 3** | **Tháng 5 với cả 86** | 2018: 64 |
| Kết thúc, `?` (còn mở) | 500 | 0 | | | |
| Kết thúc, vắng mặt (văn bản chỉ có một ngày) | 7,365 | 5,674 | trải đều | trải đều | trải đều |

- **Ít ngày được điền vào, nhưng dễ nhận ra.** Trong 12,787 ngày bắt đầu của `aired`, 12,560 (98.2%) đến từ một ngày đầy đủ trong văn bản và **227 (1.8%) từ văn bản một phần**. Trong 12,287 ngày kết thúc, **146 (1.2%)** đến từ văn bản một phần. Tổng cộng là 373 ngày (1.5% trong 25,074 ngày).
- **Mẫu giống nhau cho ngày bắt đầu và ngày kết thúc.** Với văn bản chỉ có năm, tháng là **tháng 5 trong mọi trường hợp** (105 trên 105 ngày bắt đầu, 86 trên 86 ngày kết thúc). Ngày là **9, 10, 11 hoặc 12** với 360 trong 373 ngày (96.5%). Năm là **2018** với 305 ngày (82%), ngay cả khi văn bản ghi một năm khác (`Oct, 2007` được lưu vào năm 2018); một vài ngày giữ năm của văn bản (1992, 1999, 1986), và ta không biết quy tắc chính xác. Một lần lấy dữ liệu vào 9 đến 12 tháng 5 năm 2018 sẽ cho đúng tháng này và bốn ngày này, nên giả thuyết ngày lấy dữ liệu được ủng hộ. Nó vẫn là một suy luận: ta không thấy được crawler.
- **Các ngày bắt đầu một phần còn lại là rỗng.** 354 ngày bắt đầu dạng tháng và năm và 1,114 ngày bắt đầu chỉ có năm không có ngày trong `aired` (476 - 122 và 1,219 - 105). Cộng lại chúng là 1,468 anime tìm thấy ở phần 1b. Vậy một văn bản một phần cho ra hoặc không gì cả (1,468) hoặc một ngày được điền vào (227); ta không tìm ra điều gì quyết định. 1,691 `aired` rỗng là 1,468 này cộng 223 `Not available`.
- **Ngày đầy đủ có thể lệch một năm quanh dịp năm mới.** Với 52 trong 12,560 ngày bắt đầu đầy đủ (0.41%) và 19 trong 6,467 ngày kết thúc đầy đủ (0.29%), ngày trong `aired` khác với văn bản. Mọi ví dụ đã in đều ở thời điểm giao năm: 29 đến 31 tháng 12 được lưu thành một năm sau (`Dec 30, 1996` thành `1997-12-30`, `Dec 31, 1990` thành `1991-12-31`), và 1 hoặc 2 tháng 1 được lưu thành một năm trước (`Jan 1, 1999` thành `1998-01-01`, `Jan 2, 2010` thành `2009-01-02`). Đây chính là kết quả của **năm theo tuần ISO** (tuần chứa ngày 29 tháng 12 có thể thuộc năm sau, và tuần chứa ngày 1 tháng 1 có thể thuộc năm trước). Ta kiểm tra trên 16 dòng đã in: năm ISO của ngày trong văn bản bằng năm đã lưu ở 16 trên 16 dòng, với tháng và ngày không đổi. Vậy đây có lẽ là một lỗi định dạng ở nguồn (**giả thuyết**; 55 dòng khác nhau còn lại chưa được in). Nó giải thích 4 dòng "kết thúc trước bắt đầu" mà năm lệch một.
- **Hệ quả cho bước 4.** Ngày trong `aired` chỉ dùng được làm ngày khi văn bản của `aired_string` là một ngày đầy đủ, và ngay cả khi đó năm vẫn có thể sai với ngày từ 29 tháng 12 đến 2 tháng 1. Với 373 ngày được điền vào và 1,468 ngày rỗng, hãy dùng văn bản: lấy năm khi văn bản có năm, và không gì khác. Notebook 02 (phần 4) áp dụng điều này cho 336 anime có nhãn `Not yet aired`: 54 trong 60 ngày bắt đầu sớm là ngày được điền vào.

**A6. Các cột văn bản Python (đã loại trừ).** Cả 14,478 ô của `aired`, `related`, `opening_theme` và `ending_theme` đều đọc được: 0 lỗi. Dấu `-` trong output chỉ có nghĩa là không có ví dụ.


### 7B. Bảng list (80M dòng), đọc một lần

Bảng list chưa từng được quét tìm placeholder (lần nhìn qua đầu tiên của notebook 03 bỏ qua vì kích thước). Nó được đọc một lần, mỗi lần 3M dòng, và mỗi khối chỉ cập nhật các bộ đếm. Một cột văn bản có vài nghìn giá trị khác nhau (hai cột ngày) được đếm chính xác; `my_tags`, có quá nhiều giá trị khác nhau, được đếm theo mẫu và theo một danh sách top gần đúng (300 giá trị phổ biến nhất của mỗi khối được gộp lại, nên một giá trị phổ biến sẽ được tìm thấy và số đếm của nó là cận dưới).

| Cell | Làm gì |
|---|---|
| B1 | Lượt đọc duy nhất. Chỉ in tiến độ |
| B2 | Báo cáo về `my_tags` |
| B3 | Báo cáo về hai cột ngày và về `my_last_updated` |
| B4 | Báo cáo về `anime_id` và về username của bảng list |
| B5 | Báo cáo về `my_watched_episodes` so với số tập |


In [29]:
# B1. One pass over the list table: only counters are updated, the reports are in the next cells
if RUN_LISTS:
    CHUNK = 3_000_000
    DATE_OK = r"^\d{4}-\d{2}-\d{2}$"
    REAL_DATE = r"^(?!0000)\d{4}-(?!00)\d{2}-(?!00)\d{2}$"
    CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
    EPOCH_SITE = pd.Timestamp("2004-11-01", tz="UTC").timestamp()   # MyAnimeList did not exist before
    EPOCH_END = CRAWL_END.timestamp() + 86400                       # end of 2018-05-25

    use = ["username", "anime_id", "my_watched_episodes", "my_status", "my_start_date",
           "my_finish_date", "my_last_updated", "my_tags"]
    dtypes = {"anime_id": "int32", "my_watched_episodes": "float32", "my_status": "int8", "my_last_updated": "float64"}
    episodes_of = anime.set_index("anime_id")["episodes"]
    known_ids = set(anime["anime_id"])

    rows = empty_username = orphan_rows = 0
    names, ids = set(), set()
    date_counts = {"my_start_date": Counter(), "my_finish_date": Counter()}
    both_real = finish_before_start = 0
    tag_filled = tag_html = 0
    tag_placeholder, tag_top = Counter(), Counter()
    upd_min, upd_max = float("inf"), float("-inf")
    upd_nonpositive = upd_before_site = upd_after_crawl = 0
    ep_known = watched_over = completed_rows = completed_zero = completed_less = 0

    reader = pd.read_csv(paths["UserAnimeList.csv"], usecols=use, dtype=dtypes, chunksize=CHUNK, **na_kwargs)
    for k, chunk in enumerate(reader, start=1):
        rows += len(chunk)
        empty_username += int(chunk["username"].isna().sum())
        names.update(chunk["username"].dropna().unique().tolist())
        ids.update(chunk["anime_id"].unique().tolist())
        orphan_rows += int((~chunk["anime_id"].isin(known_ids)).sum())

        for col in date_counts:
            date_counts[col].update(chunk[col].fillna("<empty>").value_counts().to_dict())
        st = chunk["my_start_date"].fillna("")
        fi = chunk["my_finish_date"].fillna("")
        real = st.str.match(REAL_DATE) & fi.str.match(REAL_DATE)
        both_real += int(real.sum())
        finish_before_start += int((real & (fi < st)).sum())

        tags = chunk["my_tags"].dropna()
        tag_filled += len(tags)
        if len(tags):
            tag_html += int(tags.str.contains(r"&#?\w+;", regex=True).sum())
            is_placeholder = tags.str.match(CANDIDATE, flags=re.IGNORECASE)
            tag_placeholder.update(tags[is_placeholder].str.lower().value_counts().to_dict())
            tag_top.update(tags.value_counts().head(300).to_dict())

        upd = chunk["my_last_updated"].dropna()
        if len(upd):
            upd_min, upd_max = min(upd_min, float(upd.min())), max(upd_max, float(upd.max()))
            upd_nonpositive += int((upd <= 0).sum())
            upd_before_site += int((upd < EPOCH_SITE).sum())
            upd_after_crawl += int((upd > EPOCH_END).sum())

        watched = chunk["my_watched_episodes"]
        total_eps = chunk["anime_id"].map(episodes_of)
        has_eps = total_eps > 0
        done = chunk["my_status"] == 2
        ep_known += int(has_eps.sum())
        watched_over += int((has_eps & (watched > total_eps)).sum())
        completed_rows += int(done.sum())
        completed_zero += int((done & (watched == 0)).sum())
        completed_less += int((done & has_eps & (watched < total_eps)).sum())

        if k % 5 == 0:
            print(f"  ... {k} chunks, {rows:,} rows")
    print(f"done: {rows:,} rows in {k} chunks")
else:
    print("Skipped (RUN_LISTS = False)")


  ... 5 chunks, 15,000,000 rows
  ... 10 chunks, 30,000,000 rows
  ... 15 chunks, 45,000,000 rows
  ... 20 chunks, 60,000,000 rows
  ... 25 chunks, 75,000,000 rows
done: 80,076,112 rows in 27 chunks


In [30]:
# B2. my_tags: how many are filled, which are placeholder-like, which are the most frequent
if RUN_LISTS:
    print(f"my_tags filled: {tag_filled:,} of {rows:,} rows ({tag_filled / rows:.1%}) | with an HTML code: {tag_html:,}")
    print("placeholder-like tags:", f"{sum(tag_placeholder.values()):,}", dict(tag_placeholder.most_common(8)))
    print("most frequent tags (approximate, a lower bound):")
    for v, c in tag_top.most_common(10):
        print(f"   {v[:40]!r} {c:,}")
else:
    print("Skipped (RUN_LISTS = False)")


my_tags filled: 4,760,289 of 80,076,112 rows (5.9%) | with an HTML code: 8,758
placeholder-like tags: 53,395 {'undefined': 33194, '-': 6194, 'null': 1957, '.': 1423, '?': 1422, '...': 1162, '---': 1002, '???': 694}
most frequent tags (approximate, a lower bound):
   'undefined' 33,172
   'Hentai' 31,295
   'Sub' 14,664
   'hentai' 14,142
   'Comedy' 13,664
   'anYme' 13,480
   'comedy' 12,967
   '*' 9,237
   'Action' 8,116
   'Dub' 7,825


In [31]:
# B3. The two date columns and my_last_updated
if RUN_LISTS:
    def summarize_dates(counter, label):
        s = pd.Series(counter, dtype="int64")
        total = int(s.sum())
        txt = pd.Series(s.index, index=s.index)
        is_format = txt.str.match(DATE_OK)
        is_empty = txt == "<empty>"
        is_zero = txt == "0000-00-00"
        other = s[~is_format & ~is_empty]
        real_format = is_format & ~is_zero
        parsed = pd.to_datetime(txt.where(real_format), format="%Y-%m-%d", errors="coerce")
        cannot = s[real_format & parsed.isna()]
        good, stamp = s[parsed.notna()], parsed[parsed.notna()]
        before_1990 = int(good[stamp < pd.Timestamp("1990-01-01")].sum())
        after_crawl = int(good[stamp > pd.Timestamp("2018-05-25")].sum())
        print(f"--- {label}: {total:,} rows, {len(s):,} distinct texts")
        print(f"  empty {int(s[is_empty].sum()):,} | 0000-00-00 {int(s[is_zero].sum()):,} | other formats {int(other.sum()):,} {other.head(5).to_dict()}")
        print(f"  right format but not a real date (for example 2010-00-00 or 2018-02-31): {int(cannot.sum()):,} {cannot.head(5).to_dict()}")
        if len(good):
            by_md = good.groupby(good.index.str[5:10]).sum().sort_values(ascending=False)
            by_year = good.groupby(stamp.dt.year).sum().sort_values(ascending=False)
            share_jan1 = by_md.get("01-01", 0) / good.sum()
            print(f"  real dates: {int(good.sum()):,} | earliest {stamp.min().date()} | latest {stamp.max().date()}")
            print(f"  before 1990: {before_1990:,} | after the crawl (2018-05-25): {after_crawl:,}")
            print("  most frequent month-day:", by_md.head(5).to_dict(), f"| share of 01-01: {share_jan1:.2%} (evenly spread: 0.27%)")
            print("  most frequent years:", by_year.head(5).to_dict())

    summarize_dates(date_counts["my_start_date"], "my_start_date")
    print()
    summarize_dates(date_counts["my_finish_date"], "my_finish_date")
    print(f"\nfinish date before start date: {finish_before_start:,} of {both_real:,} rows that have two real dates")

    print("\nmy_last_updated (seconds since 1970):")
    print("  earliest:", pd.to_datetime(upd_min, unit="s"), "| latest:", pd.to_datetime(upd_max, unit="s"))
    print(f"  <= 0: {upd_nonpositive:,} | before November 2004 (MyAnimeList did not exist): {upd_before_site:,} | after 2018-05-25: {upd_after_crawl:,}")
else:
    print("Skipped (RUN_LISTS = False)")


--- my_start_date: 80,076,112 rows, 12,536 distinct texts
  empty 0 | 0000-00-00 69,010,299 | other formats 0 {}
  right format but not a real date (for example 2010-00-00 or 2018-02-31): 425,434 {'2008-00-00': 25904, '2007-00-00': 22607, '2009-00-00': 24032, '2011-00-00': 19978, '2012-00-00': 20127}
  real dates: 10,640,379 | earliest 1693-09-25 | latest 2115-04-22
  before 1990: 1,762 | after the crawl (2018-05-25): 272
  most frequent month-day: {'04-07': 85889, '04-06': 83280, '01-10': 82223, '04-08': 81944, '04-05': 81766} | share of 01-01: 0.42% (evenly spread: 0.27%)
  most frequent years: {2015: 1392932, 2016: 1374371, 2014: 1330945, 2017: 1306770, 2013: 1190143}

--- my_finish_date: 80,076,112 rows, 12,441 distinct texts
  empty 0 | 0000-00-00 66,550,407 | other formats 0 {}
  right format but not a real date (for example 2010-00-00 or 2018-02-31): 351,987 {'2007-00-00': 18480, '2008-00-00': 21644, '2009-00-00': 20172, '2011-00-00': 17397, '2013-00-00': 17083}
  real dates: 13

In [32]:
# B4. anime_id and usernames of the list table
if RUN_LISTS:
    orphan_ids = ids - known_ids
    print(f"anime_id in the list but not in AnimeList: {len(orphan_ids):,} ids, {orphan_rows:,} rows"
          f" | anime without any list row: {len(known_ids - ids):,} of {len(known_ids):,}")

    listed = pd.Series(sorted(names))
    low = listed.str.lower()
    clash = low.duplicated(keep=False)
    print(f"\nusernames in the list table: {len(listed):,} | empty username cells: {empty_username:,}")
    print("distinct after lowercase:", f"{low.nunique():,}", "| names that differ only by capital letters:", int(clash.sum()))
    print("  examples:", listed[clash].head(10).tolist())
    print("  with a space at the start or the end:", int((listed != listed.str.strip()).sum()))

    user_names = set(users["username"])
    lower_users = set(users["username"].str.lower())
    not_found = [n for n in listed if n not in user_names]
    only_case = [n for n in not_found if n.lower() in lower_users]
    print("list usernames not found in UserList (exact text):", len(not_found),
          "| found if capital letters are ignored:", len(only_case), "| examples:", not_found[:5])
else:
    print("Skipped (RUN_LISTS = False)")


anime_id in the list but not in AnimeList: 0 ids, 0 rows | anime without any list row: 0 of 14,478

usernames in the list table: 283,045 | empty username cells: 0
distinct after lowercase: 283,045 | names that differ only by capital letters: 0
  examples: []
  with a space at the start or the end: 0
list usernames not found in UserList (exact text): 0 | found if capital letters are ignored: 0 | examples: []


In [33]:
# B5. my_watched_episodes against the episode count of the anime
if RUN_LISTS:
    print(f"rows whose anime has a known episode count (> 0): {ep_known:,} ({ep_known / rows:.1%})")
    print(f"my_watched_episodes above the episode count of the anime: {watched_over:,}")
    print(f"completed rows: {completed_rows:,} | with my_watched_episodes == 0: {completed_zero:,}"
          f" | below the episode count: {completed_less:,}")
else:
    print("Skipped (RUN_LISTS = False)")


rows whose anime has a known episode count (> 0): 79,375,560 (99.1%)
my_watched_episodes above the episode count of the anime: 207,305
completed rows: 50,445,526 | with my_watched_episodes == 0: 98,519 | below the episode count: 282,454


### Nhận xét cho phần 7B

**Cách đọc.** Lượt đọc (B1) đã đọc tất cả 80,076,112 dòng trong 27 khối (khoảng 4 phút) và các cell tiếp theo báo cáo những gì nó đã đếm.

**B2. `my_tags`: phần lớn rỗng, và các ô có giá trị chứa placeholder cùng các biến thể cách viết.**
- 4,760,289 dòng có tag (5.9%). Trong đó 8,758 dòng chứa mã HTML.
- **53,395 ô chỉ là placeholder** (1.1% các ô có giá trị): `undefined` 33,194 (62% trong số đó), `-` 6,194, `null` 1,957, `.` 1,423, `?` 1,422, `...` 1,162, `---` 1,002, `???` 694. `undefined` thậm chí là giá trị phổ biến nhất của cột. Vì sao nó xuất hiện thì ta không biết.
- Các biến thể cách viết chia nhỏ các tag thật: `Hentai` 31,295 so với `hentai` 14,142, `Comedy` 13,664 so với `comedy` 12,967. `anYme` (13,480) và `*` (9,237) xuất hiện nhiều và ta không biết chúng có nghĩa gì.
- **Giới hạn:** một ô được đếm như một tổng thể. Một ô như `undefined, Hentai` không được đếm, và danh sách top là gần đúng (cận dưới). Để đếm từng tag đơn lẻ ta phải tách mỗi ô trước (một quyết định của bước 4).

**B3. Các ngày của bảng list.**

| | `my_start_date` | `my_finish_date` |
|---|---|---|
| `0000-00-00` (không có ngày) | 69,010,299 (86.2%) | 66,550,407 (83.1%) |
| Đúng định dạng nhưng không phải ngày thật, như `2008-00-00` | 425,434 (0.5%) | 351,987 (0.4%) |
| Ngày thật | 10,640,379 (13.3%) | 13,173,718 (16.5%) |
| Ngày thật sớm nhất và muộn nhất | 1693-09-25 và 2115-04-22 | 1836-06-12 và 2101-09-26 |
| Trước 1990 | 1,762 | 6,019 |
| Sau thời điểm crawl (2018-05-25) | 272 | 1,173 |
| Tỷ lệ 1 tháng 1 | 0.42% | 0.52% |

- **Phần lớn các dòng không có ngày nào.** Chỉ 13% ngày bắt đầu và 16% ngày kết thúc là thật, nên mọi phân tích về thời điểm user xem phim chỉ dùng một phần nhỏ của bảng.
- **Năm ngày "không thật" phổ biến nhất đều có dạng `YYYY-00-00`**, tức một năm không có tháng và ngày (cách hiểu của ta: user chỉ khai năm). Các ngày không thể có như `2018-02-31` bị trộn vào số đếm này; ta chưa tách chúng ra.
- **Có những năm vô lý:** một ngày bắt đầu năm 1693, một ngày kết thúc năm 1836, và các ngày lên đến 2115. Đó là lỗi gõ. 272 ngày bắt đầu và 1,173 ngày kết thúc sau thời điểm crawl cũng không thể là ngày xem thật (crawl kết thúc khoảng 2018-05-25).
- **1 tháng 1 không phải ngày mặc định ở đây.** Nó chỉ cao gấp 1.5 đến 2 lần phần chia đều (0.27%), xa mức 16 lần thấy ở `birth_date`.
- **Một số ngày cao gấp khoảng 3 lần** và đó không phải 1 tháng 1: ngày bắt đầu vào 5 đến 8 tháng 4 và 10 tháng 1 (82,000 đến 86,000 dòng mỗi ngày, so với khoảng 29,000 nếu các ngày phân bố đều), ngày kết thúc vào 27 đến 30 tháng 3 (100,000 đến 114,000 mỗi ngày, so với khoảng 36,000) và 27 tháng 9. **Giả thuyết, chưa kiểm tra:** các mùa anime (series mới bắt đầu vào đầu tháng 1 và tháng 4, mùa đông kết thúc vào cuối tháng 3), nên các đỉnh này sẽ là thói quen xem thật chứ không phải placeholder.
- **Ngày kết thúc nằm trước ngày bắt đầu trong 25,853 dòng** (0.31% trong 8,264,175 dòng có hai ngày thật).
- **`my_last_updated` đúng bằng 0 (1970-01-01) trong 482,508 dòng (0.6%)**, nên nó là giá trị thiếu. (Không có giá trị nào nằm giữa năm 1970 và tháng 11 năm 2004, và giá trị sớm nhất là 0.) Giá trị muộn nhất là 2018-05-22 08:01:08 và không giá trị nào sau thời điểm crawl, khớp với ước tính của notebook 02.

**B4. `anime_id` và username của bảng list (đã loại trừ).**
- Mọi `anime_id` của list đều tồn tại trong `AnimeList` (0 thiếu), và mỗi anime trong 14,478 anime đều có ít nhất một dòng.
- List chứa 283,045 username. Không cái nào rỗng, không cái nào có khoảng trắng ở rìa, không cái nào chỉ khác cái khác ở chữ hoa, và **cả 283,045 đều được tìm thấy chính xác trong `UserList`**. Vậy phép join trên `username` là an toàn. (302,675 user trừ 283,045 cho ra 19,630 user không có dòng của notebook 03.)

**B5. Số tập đã xem so với số tập của anime.**
- Số tập của anime được biết (lớn hơn 0) với 79,375,560 dòng (99.1%).
- **207,305 dòng (0.26%) có số tập đã xem nhiều hơn số tập anime có.**
- Trong 50,445,526 dòng `completed` (63.0% tổng số dòng), **98,519 dòng có 0 tập đã xem** (0.20%) và **282,454 dòng có số tập đã xem ít hơn số tập anime có** (0.56%).
- Các lý do có thể (giả thuyết, chưa kiểm tra): số tập của một series dài đã thay đổi sau khi user nhập, hoặc user không cập nhật bộ đếm. Dù thế nào `my_watched_episodes` cũng không chính xác với khoảng 1% số dòng.

**Điều này thay đổi gì.**
- Đã loại trừ (không cần hành động): số placeholder, username làm khóa join (cả hai bảng), `anime_id` mồ côi, anime trùng lặp, lỗi đọc của các cột văn bản Python, khoảng của `score`, `scored_by` so với `members`.
- Các vấn đề mới cho bước 4: các dòng 23 đến 30 của bảng tóm tắt bên dưới.
- **Vẫn còn mở:** `undefined`, `anYme` và `*` trong `my_tags` có nghĩa là gì; các đỉnh vào 5 đến 8 tháng 4 và 27 đến 30 tháng 3 có phải là các mùa anime không.


## Tóm tắt bước 3: các vấn đề tìm thấy, làm đầu vào cho bước 4

Mọi thứ dưới đây được đo trên dữ liệu thô. **Không có gì bị thay đổi.** Cột cuối chỉ là đề xuất, và nhóm quyết định ở bước 4.

| # | Vấn đề | Quy mô | Ở đâu | Đề xuất |
|---|---|---|---|---|
| 1 | Các từ mà pandas giấu thành giá trị thiếu (`rating = None`, `location = N/A`, username `null`) | 544, 26 và 1 ô | Phần 1 | Giữ cách nạp an toàn. Quyết định `None` và `N/A` có được tính là thiếu không |
| 2 | Từ placeholder trong bảng anime (`source`, `broadcast`, `duration`, `aired_string`, `type`) | 4,210 / 2,241 / 405 / 223 / 30 | Phần 2 | Biến thành giá trị thiếu, và báo cáo số đếm |
| 3 | Các cột anime phần lớn rỗng (`background`, `licensor`, `premiered`, `broadcast`) | 70% đến 93% | Phần 2 | Không dùng làm đặc trưng |
| 4 | `score = 0` nghĩa là không có điểm | 354 anime (308 không có người chấm, 46 anime chưa phát sóng có ít người chấm) | Phần 2 và 6 | Biến thành giá trị thiếu |
| 5 | Ít người chấm | 4,789 anime (33.1%) có dưới 100 | Phần 2 | Chọn một ngưỡng và báo cáo số bị loại |
| 6 | `rank` và `popularity` bằng 0, `episodes = 0` | 9, 9 và 504 anime | Phần 2 và 6 | Coi là giá trị thiếu |
| 7 | `access_rank` rỗng | 100% | Phần 3 | Bỏ cột |
| 8 | Thiếu gender, ngày sinh và location | 28.0% / 44.3% / 48.2% | Phần 3 | Phân tích những user có khai, và nói rõ điều đó |
| 9 | Ngày sinh không thể có | 1,724 tuổi dưới 8 hoặc trên 80 (842 người già sinh năm 1930, có lẽ là mặc định), 8 văn bản giả, các năm 1800 và 2137 | Phần 3 và 6 | Coi 1930 là thiếu, giữ một khoảng tuổi hợp lý, báo cáo số bị loại |
| 10 | `last_online` chứa năm 1900 | 89,283 user (29.5%), tất cả đúng vào năm 1900 | Phần 3 và 6 | Không dùng `last_online` làm ngày |
| 11 | `stats_mean_score = 0` nghĩa là không có điểm | 30,922 user | Phần 3 | Biến thành giá trị thiếu |
| 12 | Giá trị lớn nhất không thể có trong `user_days_spent_watching` và `stats_episodes` | 2,601,351 ngày và 243,794,900 tập; 204 user trên 100,000 tập, 42 trên 3,650 ngày | Phần 3 và 6 | Chặn ở phân vị 99.9 hoặc loại bỏ |
| 13 | Số khai báo khác với các dòng trong list | 0.1% đến 1.8% số user ở mỗi trạng thái; 6 user khai 0 nhưng có dòng | Phần 4 | Chỉ dùng số khai báo làm bản tóm tắt |
| 14 | User không có dòng | 19,630 user: 19,583 tài khoản rỗng, 47 có hoạt động, 8 khai có anime | Phần 5 | Bỏ các tài khoản rỗng khỏi phân tích list, báo cáo 47 user có hoạt động và 8 user khai có anime (một số nằm ở cả hai nhóm) |
| 15 | Hai username dùng chung một `user_id` (2 trường hợp), với cùng một list được lưu hai lần | 1,059 dòng list bị trùng | Notebook 02, phần 4 | Giữ tên có profile đầy đủ |
| 16 | `duration` viết bằng giây | 313 anime (405 anime khác là `Unknown`) | Phần 6 | Đọc `sec` thành phút chia cho 60; chỉ 405 `Unknown` là thiếu |
| 17 | Danh sách rỗng và dictionary rỗng dùng làm "không có giá trị" | `opening_theme` 9,784 (67.6%), `ending_theme` 8,807 (60.8%), `related` 4,515 (31.2%), `aired` 1,691 (11.7%) | Phần 1a, 1b | Coi là thiếu (hoặc là "không có"). Với 1,691 `aired` rỗng, 1,468 có năm hoặc tháng trong `aired_string` (phần 1b): đọc năm phát hành từ `aired_string` |
| 18 | `broadcast` có thứ trong tuần mà không có giờ (`Sundays at Unknown`), và `Not scheduled once per week` | 78 và 65 anime | Phần 1a | Coi giờ là thiếu |
| 19 | Placeholder do user gõ trong `location` | 309 ô (0.2%), cộng 78 ô nằm trong văn bản dài hơn | Phần 1a | Lập một danh sách cho `location`, và gom nhóm phần còn lại |
| 20 | `birth_date` với 1 tháng 1 làm ngày mặc định | 7,353 trong 168,741 user = 4.36% (phân bố đều: 0.27%, khoảng 462 user); tỷ lệ giảm từ 6.5% (sinh 1985) xuống 1.9% (sinh 2000) | Phần 1b | Chỉ dùng năm sinh; không dùng tháng hay ngày |
| 21 | Mã HTML như `&#039;` và `&amp;` trong văn bản | 11 cột: `background` 316 (29.9%), `title_synonyms` 739 (8.3%), `title_english` 409 (7.1%), `title` 202, `studio` 130, `producer` 168, theme, `related`, `location` 40 | Phần 1b | Giải mã mọi cột văn bản bằng `html.unescape` |
| 22 | Ngày bắt đầu được lưu với năm, tháng hoặc ngày điền vào (cụm 2018 vào ngày 11 tìm thấy ở 1a) | Ngày 11 nói chung không phải mặc định (457 trong 12,787 ngày bắt đầu, trung bình 412 mỗi ngày). Các đỉnh 2018 vào ngày 11 (`2018-05` 60, `2018-07` 27, `2018-10` 20) là các ngày được điền vào cho văn bản một phần (phần 7, A5b); tổng cộng là 227 ngày bắt đầu và 146 ngày kết thúc, phần 7A, A5c. Các ngày 21 (678) và 25 (808) vẫn còn mở | Phần 1b, 7A | Không dùng ngày hay năm của một ngày mà `aired_string` chỉ có một phần; đọc năm phát hành từ văn bản |
| 23 | `my_tags`: placeholder, biến thể cách viết và mã HTML | 4,760,289 dòng có giá trị (5.9%); 53,395 ô placeholder (`undefined` 33,194); `Hentai` 31,295 so với `hentai` 14,142; 8,758 dòng có mã HTML | B2 | Coi placeholder là rỗng, đổi chữ thường, giải mã, và tách ô thành từng tag đơn lẻ trước khi đếm |
| 24 | `my_start_date` và `my_finish_date` bằng `0000-00-00` | 69,010,299 (86.2%) và 66,550,407 (83.1%) | B3 | Coi là thiếu; chỉ khoảng 13% đến 16% số dòng có ngày thật |
| 25 | Ngày chỉ có năm (`YYYY-00-00`) hoặc năm không thể có | 425,434 và 351,987 chỉ có năm; năm sớm nhất 1693 và 1836, muộn nhất 2115 và 2101; 272 và 1,173 sau thời điểm crawl | B3 | Giữ năm của các ngày chỉ có năm; coi các ngày ngoài một khoảng hợp lý là thiếu (quyết định các giới hạn) |
| 26 | Ngày kết thúc trước ngày bắt đầu | 25,853 trong 8,264,175 dòng có hai ngày thật (0.31%) | B3 | Coi cả hai ngày là không đáng tin ở các dòng này |
| 27 | `my_last_updated` bằng 0 | 482,508 dòng (0.6%) | B3 | Coi là thiếu |
| 28 | Cách viết của `location`, khoảng trắng ở rìa | 55,283 văn bản thô, 49,616 sau khi chuẩn hóa; 4,560 ô có khoảng trắng ở rìa | A2, A3 | Cắt khoảng trắng, đổi chữ thường và gộp khoảng trắng, rồi gom nhóm địa điểm theo quy tắc |
| 29 | Các ngày `aired` được điền vào cho một `aired_string` một phần (năm 2018 với 82%, tháng 5 với mọi văn bản chỉ có năm, ngày 9 đến 12 với 96.5%), và các mâu thuẫn chúng tạo ra | 373 ngày: 227 trong 12,787 ngày bắt đầu (1.8%) và 146 trong 12,287 ngày kết thúc (1.2%). Chúng giải thích tất cả 16 anime "Finished Airing" có ngày bắt đầu sau thời điểm crawl, phần lớn trong 32 trường hợp "kết thúc trước bắt đầu" (21 trường hợp theo mắt thường) và 4 trong 6 anime "Currently Airing" có ngày kết thúc trước thời điểm crawl | 7A (A5, A5b, A5c) | Đánh dấu các ngày này. Chỉ dùng một ngày khi văn bản đầy đủ; với văn bản một phần lấy năm từ văn bản. Đã làm cho 60 anime có ngày sớm của notebook 02: 54 ngày của chúng là ngày bịa ra (notebook 02, phần 4) |
| 30 | Xem nhiều tập hơn số tập anime có; `completed` với ít tập hoặc không có tập nào | 207,305 dòng; 98,519 completed với 0; 282,454 completed với ít hơn | B5 | Không coi `my_watched_episodes` là chính xác; định nghĩa một quy tắc cho các dòng này |
| 31 | Các ngày đầy đủ trong `aired` lệch một năm quanh dịp năm mới (trông giống năm theo tuần ISO) | 52 trong 12,560 ngày bắt đầu đầy đủ (0.41%) và 19 trong 6,467 ngày kết thúc đầy đủ (0.29%); 29 đến 31 tháng 12 được lưu thành một năm sau, 1 đến 2 tháng 1 thành một năm trước | 7A (A5c) | Không tin năm của một ngày từ 29 tháng 12 đến 2 tháng 1; đọc nó từ văn bản |

**Đã loại trừ ở phần 7:** số placeholder, username (không có xung đột chữ hoa, không có khoảng trắng ở rìa, mọi username của list đều tìm thấy trong `UserList`), `anime_id` mồ côi, anime trùng lặp, lỗi đọc của bốn cột văn bản Python, khoảng của `score`, `scored_by` so với `members`.

**Điều không thay đổi.** Bảng anime và bảng user không có khóa lặp, phép join trên `username` không có dòng mồ côi, và số khai báo khớp với bảng list với hơn 98% số user. Dữ liệu dùng được sau khi xử lý các vấn đề trên.
